In [ ]:
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
sns.set_style('whitegrid')
from sklearn.metrics import confusion_matrix , classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Flatten , Conv2D , MaxPooling2D , Dropout , Activation , BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam , Adamax
from tensorflow.keras import regularizers

#Warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import tensorflow as tf
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet121, ResNet50V2
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet169, MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy

import tensorflow as tf
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras import layers


## Modality 1

In [ ]:
X_train_h = np.load('x_train.npy')
y_train_h = np.load('y_train.npy')
X_test_h = np.load('x_test.npy')
y_test_h = np.load('y_test.npy')

X_val_h = np.load('x_val.npy')
y_val_h = np.load('y_val.npy')


X_train_h.shape, y_train_h.shape, X_test_h.shape, y_test_h.shape, X_val_h.shape, y_val_h.shape


In [ ]:
random_indices = np.random.choice(1001, 810, replace=False)

X_test_h1 = X_test_h[random_indices]
y_test_h1 = y_test_h[random_indices]

X_test_h1.shape, y_test_h1.shape, X_test_h.shape, y_test_h.shape

In [ ]:
random_indices = np.random.choice(1002, 648, replace=False)

X_val_h1 = X_val_h[random_indices]
y_val_h1 = y_val_h[random_indices]

X_test_h1.shape, y_test_h1.shape, X_test_h.shape, y_test_h.shape, X_val_h1.shape, y_val_h1.shape

In [ ]:
X_train_h.shape, y_train_h.shape, X_test_h.shape, y_test_h.shape

## Modality 2

In [ ]:
X_train_s = np.load('X.npy')
y_train_s = np.load('y.npy')

X_train_s.shape, y_train_s.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)

X_train_s.shape,X_test_s.shape, y_train_s.shape,y_test_s.shape, y_val_s.shape,y_val_s.shape

In [ ]:
import numpy as np
import cv2

def rotate_image(image, angle):
    """
    Rotate the image by the specified angle.
    """
    center = tuple(np.array(image.shape[1::-1]) / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_image = cv2.warpAffine(image, rotation_matrix, image.shape[1::-1], flags=cv2.INTER_LINEAR)
    return rotated_image

def translate_image(image, tx, ty):
    """
    Translate the image by the specified translation parameters.
    """
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    translated_image = cv2.warpAffine(image, translation_matrix, image.shape[1::-1])
    return translated_image

def apply_gaussian_blur(image, kernel_size=3):
    """
    Apply Gaussian Blur to the image to reduce noise and improve generalization.
    """
    blurred_image = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return blurred_image

# Augmentation parameters
rotation_angles = [20]
translations = [(5, 5)]
kernel_sizes = [3]  # Gaussian Blur kernel sizes

augmented_X_train = []
augmented_y_train = []

for image, label in zip(X_train_s, y_train_s):
    # Augment with rotations
    for angle in rotation_angles:
        rotated_image = rotate_image(image, angle)
        augmented_X_train.append(rotated_image)
        augmented_y_train.append(label)

    # Augment with translations
    for tx, ty in translations:
        translated_image = translate_image(image, tx, ty)
        augmented_X_train.append(translated_image)
        augmented_y_train.append(label)

    # Augment with Gaussian Blur
    for kernel_size in kernel_sizes:
        blurred_image = apply_gaussian_blur(image, kernel_size)
        augmented_X_train.append(blurred_image)
        augmented_y_train.append(label)

# Convert lists to numpy arrays
augmented_X_train = np.array(augmented_X_train)
augmented_y_train = np.array(augmented_y_train)

# Shuffle the data
shuffle_indices = np.random.permutation(len(augmented_X_train))
augmented_X_train = augmented_X_train[shuffle_indices]
augmented_y_train = augmented_y_train[shuffle_indices]
augmented_X_train.shape, augmented_y_train.shape


In [ ]:
random_indices = np.random.choice(7773, 5421, replace=False)

augmented_X_train = augmented_X_train[random_indices]
augmented_y_train = augmented_y_train[random_indices]

augmented_X_train.shape, augmented_y_train.shape


In [ ]:
X_train_s = np.concatenate((X_train_s, augmented_X_train), axis=0)
y_train_s = np.concatenate((y_train_s, augmented_y_train), axis=0)
X_train_s.shape, y_train_s.shape

In [ ]:
X_train_s.shape, y_train_s.shape

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers


def multi_kernel_groupwise_conv2(x, filters, groups=16, strides=1):
    # 1x1 Group-wise Convolution
    conv1x1 = layers.DepthwiseConv2D(kernel_size=1, strides=strides, padding="same")(x)

    # 3x3 Group-wise Convolution
    conv3x3 = layers.DepthwiseConv2D(kernel_size=5, strides=strides, padding="same")(x)

    # 5x5 Group-wise Convolution
    conv5x5 = layers.DepthwiseConv2D(kernel_size=7, strides=strides, padding="same")(x)

    # Depthwise 3x3 Group-wise Convolution
    #depthwise3x3 = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x)
    
    # Concatenate all outputs along the channel axis
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5])
    x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same")(x1)

    x = layers.Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(x)
    x = layers.Add()([x, x1])
    x = layers.Activation('relu')(x)
    
    return x

import tensorflow as tf
from tensorflow.keras import layers

class MultiKernelGroupwiseConv2(layers.Layer):
    def __init__(self, filters, groups=16, strides=1, **kwargs):
        super(MultiKernelGroupwiseConv2, self).__init__(**kwargs)
        self.filters = filters
        self.groups = groups
        self.strides = strides

        # 1x1 Group-wise Convolution
        self.conv1x1 = layers.DepthwiseConv2D(kernel_size=1, strides=strides, padding="same")

        # 3x3 Group-wise Convolution
        self.conv3x3 = layers.DepthwiseConv2D(kernel_size=5, strides=strides, padding="same")

        # 5x5 Group-wise Convolution
        self.conv5x5 = layers.DepthwiseConv2D(kernel_size=7, strides=strides, padding="same")

        # Final 1x1 Group-wise Convolution
        self.final_conv = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same")

        # Shortcut Path
        self.shortcut_conv = layers.Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')

        # Activation Function
        self.activation = layers.Activation('relu')

    def call(self, inputs):
        conv1x1 = self.conv1x1(inputs)
        conv3x3 = self.conv3x3(inputs)
        conv5x5 = self.conv5x5(inputs)

        # Concatenate along the channel axis
        x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5])
        x1 = self.final_conv(x1)

        # Shortcut connection
        x = self.shortcut_conv(inputs)
        x = layers.Add()([x, x1])
        x = self.activation(x)

        return x


# Custom Attention Block with Global Pooling
import tensorflow as tf
from tensorflow.keras import layers

class GlobalMinPooling2D(layers.Layer):
    def __init__(self, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2])

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        return config




# Residual Attention Block (Using Spatial Attention)


import tensorflow as tf
from tensorflow.keras import layers

def multi_kernel_groupwise_conv1(x, filters, groups=8, strides=2):
    """ Advanced Multi-Kernel Groupwise Convolution Block (Without Attention, Optimized) """

    # Multi-Kernel Parallel Convolutions with Different Receptive Fields
    conv1x1 = layers.Conv2D(filters // 4, kernel_size=1, padding="same")(x)
    conv3x3 = layers.DepthwiseConv2D(kernel_size=3, padding="same")(x)
    conv5x5 = layers.DepthwiseConv2D(kernel_size=5, padding="same")(x)
    conv_dilated = layers.DepthwiseConv2D(kernel_size=3, dilation_rate=2, padding="same")(x)

    # Feature Fusion via Concatenation
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5, conv_dilated])

    # Channel Shuffle for Better Feature Mixing
    def channel_shuffle(x, groups):
        batch, height, width, channels = tf.unstack(tf.shape(x))
        x = tf.reshape(x, [-1, height, width, groups, channels // groups])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [-1, height, width, channels])
        return x

    x1 = layers.Lambda(lambda x: channel_shuffle(x, groups))(x1)

    # Depthwise + Grouped Convolutions Hybrid
    x1 = layers.DepthwiseConv2D(kernel_size=3, padding="same")(x1)
    x1 = layers.Conv2D(filters, kernel_size=1, padding="same")(x1)

    # Downsampling x1
    x1 = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x1)

    # **Fix: Ensure x has the same number of channels as x1**
    x = layers.Conv2D(filters, kernel_size=1, padding="same")(x)
    x = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x)

    # Residual Connection
    x = layers.Add()([x, x1])
    x = layers.Activation("relu")(x)
    
    return x

import tensorflow as tf
from tensorflow.keras import layers



import tensorflow as tf
from tensorflow.keras import layers

## MSGCA
'''def multi_kernel_groupwise_conv(x, filters, groups=8, strides=1, use_se=True):
    # GPC for MSGDC
    conv1x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, strides=strides, padding="same", use_bias=False)(x)
    

    # DDC (Multi-scale receptive fields) for MSGDC
    conv3x3 = layers.DepthwiseConv2D(kernel_size=3, dilation_rate=1, strides=strides, padding="same", use_bias=False)(x)
    

    # DWC (Multi-scale receptive fields) for MSGDC
    conv5x5 = layers.DepthwiseConv2D(kernel_size=5, strides=strides, padding="same", use_bias=False)(x)
    

    # Concatenation and 1x1 Fusion for MSHC
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5])

    def channel_shuffle(x, groups):
        batch, height, width, channels = tf.unstack(tf.shape(x))
        x = tf.reshape(x, [-1, height, width, groups, channels // groups])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [-1, height, width, channels])
        return x

    x1 = layers.Lambda(lambda x: channel_shuffle(x, groups))(x1)
    
    x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same", use_bias=False)(x1)
    x1 = layers.BatchNormalization()(x1)

    # CA layer

    GMN = tf.reduce_min(inputs, axis=[1, 2])(x1)
    GMP = tf.reduce_max(inputs, axis=[1, 2])(x1)
    GAP = tf.reduce_mean(inputs, axis=[1, 2])(x1)
    GSP = tf.reduce_suminputs, axis=[1, 2])(x1)

    
    channel_add = GMN + GAP + GMP + GSP

    channel_sub = GMP - GAP - GMN
    
    channel_info = channel_add + channel_sub

    se = layers.Dense(filters // 16, activation='relu', use_bias=False)(div_channel)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)

    se = layers.Reshape((1, 1, filters))(se)
    
    self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.HeNormal(), trainable=True, name='global_scale')
    se *= self.global_scale
    se = layers.Activation('sigmoid')(se)
    
    
    x1 = layers.Multiply()([x1, se])

    
    # Residual Connection
    x = layers.Conv2D(filters=filters, kernel_size=1, strides=strides, padding='same', use_bias=False)(x)
    

    x = layers.Add()([x, x1])
    x = layers.Activation('relu')(x)  # GELU activation for better convergence

    return x
'''




import tensorflow as tf
from tensorflow.keras import layers



def channel_shuffle(tensor, groups: int):
    """Shuffle channels (ShuffleNet-style)."""
    b, h, w, c = tf.unstack(tf.shape(tensor))
    tensor = tf.reshape(tensor, [b, h, w, groups, c // groups])
    tensor = tf.transpose(tensor, [0, 1, 2, 4, 3])
    tensor = tf.reshape(tensor, [b, h, w, c])
    return tensor


# EMILA
class MultiKernelGroupWiseConv(layers.Layer):
    """
    Multi-kernel / group-wise conv block with channel-shuffle,
    global min/max/avg/sum attention, and residual connection.
    """

    def __init__(self,
                 filters: int,
                 groups: int = 2,
                 strides: int = 1,
                 use_se: bool = True,
                 reduction: int = 4,
                 **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.groups = groups
        self.strides = strides
        self.use_se = use_se
        self.reduction = reduction

        # ── convolutions ───────────────────────────────────────────────
        # 1×1 grouped of MSGDC
        self.conv1x1 = layers.Conv2D(filters, 1,
                                     strides=strides,
                                     padding="same",
                                    # use_bias=False,
                                     groups=groups)
        self.bn1x1 = layers.BatchNormalization()

        # dw-3×3   of MSGDC
        self.dw3 = layers.DepthwiseConv2D(3, strides=strides,
                                          padding="same",
                                         # use_bias=False
                                         )
        self.bn_dw3 = layers.BatchNormalization()
        #self.pw3 = layers.Conv2D(filters, 1, padding="same", use_bias=False, groups=groups)
        #self.bn_pw3 = layers.BatchNormalization()

        # dw-5×5   of MSGDC
        self.dw5 = layers.DepthwiseConv2D(5, strides=strides,
                                          padding="same",
                                         # use_bias=False
                                         )
        self.bn_dw5 = layers.BatchNormalization()
        #self.pw5 = layers.Conv2D(filters, 1, padding="same", use_bias=False, groups=groups)
        #self.bn_pw5 = layers.BatchNormalization()

        # fusion after concat
        self.fuse = layers.Conv2D(filters, 1,
                                  padding="same",
                                 # use_bias=False, 
                                  groups=2
                                  )
        self.bn_fuse = layers.BatchNormalization()

        
        #if use_se:
        self.fc1 = layers.Dense(filters // reduction,
                                activation="relu")
        self.fc2 = layers.Dense(filters)
        self.reshape = layers.Reshape((1, 1, filters))

        # residual projection
        self.proj = layers.Conv2D(filters, 1,
                                  strides=strides,
                                  padding="same",
                                 # use_bias=False
                                 )
        self.bn_proj = layers.BatchNormalization()

        #self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.HeNormal(), trainable=True, name='global_scale')
    

        self.act = layers.Activation("relu")

    # global trainable scale requested by you
    def build(self, input_shape):
        self.global_scale = self.add_weight(
            name="global_scale",
            shape=(1, 1, 1, 1),
            initializer=tf.keras.initializers.HeNormal(),
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        # ── branch 1: grouped 1×1 ──────────────────────────────────────
        b1 = self.conv1x1(x)

        # ── branch 2: dw-3×3 ──────────────────────────────────────────
        b2 = self.dw3(x)
        #b2 = self.bn_dw3(b2)
        #b2 = self.pw3(b2)

        # ── branch 3: dw-5×5 ──────────────────────────────────────────
        b3 = self.dw5(x)
        #b3 = self.bn_dw5(b3)
        #b3 = self.pw5(b3)

        # ── concat + shuffle + fuse ───────────────────────────────────
        out = tf.concat([b1, b2, b3], axis=-1)
        out = channel_shuffle(out, self.groups)
        out = self.bn_fuse(self.fuse(out))

        # ── CA  ────────────────
        
        g_min = tf.reduce_min(out, axis=[1, 2])
        g_max = tf.reduce_max(out, axis=[1, 2])
        g_avg = tf.reduce_mean(out, axis=[1, 2])
        g_sum = tf.reduce_sum(out, axis=[1, 2])

        channel_add = g_min + g_avg + g_max + g_sum 
        channel_sub = g_max - g_avg - g_min
        channel_info = channel_add + channel_sub

        se = self.fc1(channel_info)
        se = self.fc2(se)
        se = self.reshape(se)                  # (1,1,C)
        # ← the three lines you explicitly asked for

        
        se *= self.global_scale
        se = tf.nn.sigmoid(se)

        out *= se

        

        #print('out shape:', out.shape)
        #out = self.fuse(out) # GPC

        # ── residual connection ───────────────────────────────────────
        shortcut = self.bn_proj(self.proj(x))
        out = self.act(shortcut + out)
        return out



import tensorflow as tf
from tensorflow.keras import layers

### ERLA block
def RGSA(x, filters, strides=(1, 1), use_projection=False):
    shortcut = x  # Save the original input for residual connection

    # Multi-Kernel Groupwise Convolution (First Layer)
    #x = MultiKernelGroupWiseConv(x, filters=filters, groups=16, strides=strides)

    x = MultiKernelGroupWiseConv(filters=filters, groups=16, strides=strides)(x)

    # Normalization and Activation
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Depthwise and Pointwise Convolution
    #x = MultiKernelGroupWiseConv(x, filters=filters, groups=16, strides=(1, 1))  # Strides=1 to avoid mismatch

    x = MultiKernelGroupWiseConv(filters=filters, groups=16, strides=(1, 1))(x)
    
    x = layers.Conv2D(filters, kernel_size=(1, 1), padding="same")(x)  # Pointwise conv for channel mixing
    x = layers.BatchNormalization()(x)

    # Adjust Shortcut if Needed (Ensure Matching Shape)
    if strides != (1, 1) or use_projection:
        shortcut = layers.Conv2D(filters, kernel_size=(1, 1), strides=strides, padding="same")(shortcut)  # Downsampling shortcut
        shortcut = layers.BatchNormalization()(shortcut)

    # Residual Connection (Ensure Same Shape)
    x = layers.Add()([x, shortcut])

    # Final Activation
    x = layers.Activation('relu')(x)

    return x


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class GlobalMinPooling2D(layers.Layer):
    def __init__(self, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2])

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        return config


## EMCAM
class DeeperGlobalLocalAttentionLayer1(layers.Layer):
    def __init__(self, units, activation='sigmoid', dropout_rate=0.2, groups=16, use_scale=True, **kwargs):
        super(DeeperGlobalLocalAttentionLayer1, self).__init__(**kwargs)
        self.units = units
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_scale = use_scale
        self.groups = groups

    def build(self, input_shape):
        input1, input2 = input_shape
        _, _, _, channels1 = input1
        _, _, _, channels2 = input2
        
        self.alpha1 = self.add_weight(shape=(1, 1, 1, channels1), initializer='ones', trainable=True)
        self.alpha2 = self.add_weight(shape=(1, 1, 1, channels2), initializer='ones', trainable=True)
        
        self.global_min_pooling = GlobalMinPooling2D()
        self.global_avg_pooling = layers.GlobalAveragePooling2D()
        self.global_max_pooling = layers.GlobalMaxPooling2D()
        
        self.global_attention = layers.Dense(units=self.units, activation=self.activation)
        
        self.local_conv1 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.local_conv2 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        
        #self.min_pooling = MinPooling2D(pool_size = 1)
        #self.min_pooling1 = MinPooling2D(pool_size = 1)
        self.avg_pooling = layers.AveragePooling2D(pool_size = 1)
        self.max_pooling = layers.MaxPooling2D(pool_size = 1)
        
        self.sub = layers.Subtract()
        self.mod_scale1 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True)
        self.mod_scale2 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True)
        self.mod_scale3 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True)
        
        self.msgdc_conv1x1 = layers.Conv2D(
            self.units, 1, padding="same", use_bias=False, groups=self.groups
        )
        self.msgdc_dw3 = layers.DepthwiseConv2D(
            3, padding="same", use_bias=False
        )
        self.msgdc_dw5 = layers.DepthwiseConv2D(
            5, padding="same", use_bias=False
        )
        self.msgdc_fuse = layers.Conv2D(
            self.units, 1, padding="same", use_bias=False
        )
        
        if self.use_scale:
            self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale')
            self.local_scale = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale')
            
            self.global_scale2 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale2')
            self.local_scale2 = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale2')
        
        super(DeeperGlobalLocalAttentionLayer1, self).build(input_shape)

    def MFIFA(self, x):
            input_shape1 = tf.shape(x)
            flattened_inputs1 = tf.reshape(x, [-1, input_shape1[-1]])  # Flatten along the last axis
            dct_transformed1 = tf.signal.dct(flattened_inputs1, type=2, norm='ortho')
            dct_transformed1 = tf.reshape(dct_transformed1, input_shape1)  # Reshape back to original dimensions
            
            High1 = self.global_max_pooling(dct_transformed1)   ## High 1
            High1 = self.global_attention(High1)
            High1 = tf.expand_dims(tf.expand_dims(High1, 1), 1)
    
            Low1 = self.global_avg_pooling(dct_transformed1)  ## Low 1
            Low1 = self.global_attention(Low1)
            Low1 = tf.expand_dims(tf.expand_dims(Low1, 1), 1)
    
            
            Low2 = self.global_min_pooling(dct_transformed1)   ## Low 2
            Low2 = self.global_attention(Low2)
            Low2 = tf.expand_dims(tf.expand_dims(Low2, 1), 1)
            
            High2 = High1 - Low1
            
            High3 = dct_transformed1 - Low2
    
            High = High1 + High2 + High3
            Low = Low1 + Low2
            Mean = High - Low

            #High = tf.expand_dims(tf.expand_dims(High, 1), 1)
            #Low  = tf.expand_dims(tf.expand_dims(Low, 1), 1)
            #Mean = tf.expand_dims(tf.expand_dims(Mean, 1), 1)
    
            High *= self.mod_scale1 # Modulation
            Low *= self.mod_scale2 # Modulation
            Mean *= self.mod_scale3 # Modulation
            
            return High, Low, Mean

    
    def _multi_kernel_groupwise_conv(self, x):
        """MSGDC block, *re-using* layers defined in build()."""
        c1 = self.msgdc_conv1x1(x)
        c3 = self.msgdc_dw3(x)
        c5 = self.msgdc_dw5(x)

        concat = tf.concat([c1, c3, c5], axis=-1)
        return self.msgdc_fuse(concat)

        

    def call(self, inputs, training=None):
        input1, input2 = inputs
        
        # MFIFA

        High_1, Low_1, Mean_1 = self.MFIFA(input1) # for input modality 1
        High_2, Low_2, Mean_2 = self.MFIFA(input2) # for input modality 2

        
        High = High_1 + High_2
        Low = Low_1 + Low_2
        Mean = Mean_1 + Mean_2
        
        attention = tf.sigmoid(High + Low + Mean)  # MFIFA Attention map generation
        
        
        # Deeper Spatial Attention
        
        local_attention1 = self._multi_kernel_groupwise_conv(input1) #self.local_conv1(input1)
        local_attention1_mean = tf.reduce_mean(local_attention1, axis=[1, 2], keepdims=True)
        local_attention1_max = tf.reduce_max(local_attention1, axis=[1, 2], keepdims=True)
        local_attention1 = local_attention1_max + local_attention1_mean

        local_attention2 = self._multi_kernel_groupwise_conv(input2) #self.local_conv1(input1)
        local_attention2_mean = tf.reduce_mean(local_attention2, axis=[1, 2], keepdims=True)
        local_attention2_max = tf.reduce_max(local_attention2, axis=[1, 2], keepdims=True)
        local_attention2 = local_attention2_max + local_attention2_mean
        
        
        
        local_attention1_1 = self._multi_kernel_groupwise_conv(local_attention1)
        local_attention1_1_mean = tf.reduce_mean(local_attention1_1, axis=[1, 2], keepdims=True)
        local_attention1_1_max = tf.reduce_max(local_attention1_1, axis=[1, 2], keepdims=True)
        local_attention1 = local_attention1 + local_attention1_1_mean + local_attention1_1_max  + local_attention2 # Skip with Cross-modal Interactions

        
        
        local_attention2_1 = self._multi_kernel_groupwise_conv(local_attention2)
        local_attention2_1_mean = tf.reduce_mean(local_attention2_1, axis=[1, 2], keepdims=True)
        local_attention2_1_max = tf.reduce_max(local_attention2_1, axis=[1, 2], keepdims=True)
        local_attention2 = local_attention2 + local_attention2_1_mean + local_attention2_1_max  + local_attention1 # Skip with Cross-modal Interactions

        
        #local_attention2 = local_attention2_ + local_attention2__

        # Scale Global and Spatial Attention
    
        
        local_attention1 *= self.local_scale # Modulation
        local_attention2 *= self.local_scale2 # Modulation

        # Combine 
        attention1 = tf.sigmoid(local_attention1 + local_attention2) # EMSCA Attention map generation

        attention *= self.global_scale # Modulation
        attention1 *= self.local_scale # Modulation
        
        attention2 = tf.sigmoid(attention + attention1) # dUal domain Parallel Fusion Attention Map Generation via EMCAM
        
        
        return attention2 

    def get_config(self):
        config = super(DeeperGlobalLocalAttentionLayer1, self).get_config()
        config.update({'units': self.units, 'activation': self.activation, 'dropout_rate': self.dropout_rate,
                       'use_scale': self.use_scale})
        return config


## EMCAM Attention Map Integration Into Input modalities
class DeeperAttentionLayer1(layers.Layer):
    def __init__(self, units=64, use_scale=True, **kwargs):
        super(DeeperAttentionLayer1, self).__init__(**kwargs)
        self.units = units
        self.use_scale = use_scale

    def build(self, input_shape):
        input1, input2 = input_shape
        _, H, W, C1 = input1
        _, H, W, C2 = input2
        
        self.alpha3 = self.add_weight(shape=(1, 1, 1, C1), initializer='ones', trainable=True, name='alpha3')
        self.alpha4 = self.add_weight(shape=(1, 1, 1, C2), initializer='ones', trainable=True, name='alpha4')
        self.add = layers.Add()
        self.deeper_global_local_attention = DeeperGlobalLocalAttentionLayer1(units=self.units, activation='sigmoid', 
                                                                              dropout_rate=0.2,
                                                                              use_scale=self.use_scale)
        super(DeeperAttentionLayer1, self).build(input_shape)

    def call(self, inputs, training=None):
        input1, input2 = inputs
        attention2 = self.deeper_global_local_attention(inputs, training=training)
        attention_feature1 = input1 * attention2 * self.alpha3 # Attention Map Integration Into Input modality 1
        attention_feature2 = input2 * attention2 * self.alpha4 # Attention Map Integration Into Input modality 2
        
        #attention_feature = self.add([attention_feature1, attention_feature2])
        return attention_feature1, attention_feature2

    def get_config(self):
        config = super(DeeperAttentionLayer1, self).get_config()
        config.update({'units': self.units, 'use_scale': self.use_scale})
        return config


## MILA ##

In [ ]:
import math
import tensorflow as tf
from tensorflow.keras import layers


# ---------------------------------------------------------------------
# Utility
# ---------------------------------------------------------------------

def _safe_conv_group_count(input_channels, filters, requested_groups):
    """
    Safe group count for grouped Conv2D.

    Keras Conv2D(groups=g) requires:
        input_channels % g == 0
        filters % g == 0
    """
    if input_channels is None or filters is None:
        raise ValueError("Input channels and filters must be statically known.")

    input_channels = int(input_channels)
    filters = int(filters)
    requested_groups = int(requested_groups)

    if requested_groups <= 1:
        return 1

    requested_groups = min(requested_groups, input_channels, filters)

    return max(
        1,
        math.gcd(math.gcd(input_channels, filters), requested_groups)
    )


# ---------------------------------------------------------------------
# Ghost convolution replacement for MILA GPC / fusion projection
# ---------------------------------------------------------------------

class GhostConv2D(layers.Layer):
    """
    GhostConv2D for replacing expensive point-wise/group point-wise Conv2D.

    It first generates intrinsic feature maps using a primary convolution,
    then generates cheap ghost feature maps using depth-wise convolution.

    Output shape:
        [B, H, W, filters]

    For MILA:
        - use kernel_size=1 to replace GPC / 1x1 projection
        - use groups=group_count to keep group point-wise behavior
        - use cheap_kernel_size=3 for standard ghost-style cheap features
          or cheap_kernel_size=1 if you want strictly point-wise cheap features.
    """
    def __init__(
        self,
        filters,
        kernel_size=1,
        ratio=2,
        cheap_kernel_size=3,
        strides=1,
        padding="same",
        groups=1,
        activation=None,
        use_bias=False,
        kernel_initializer="glorot_uniform",
        bias_initializer="zeros",
        **kwargs
    ):
        super(GhostConv2D, self).__init__(**kwargs)

        if filters < 1:
            raise ValueError("filters must be >= 1.")
        if ratio < 1:
            raise ValueError("ratio must be >= 1.")

        self.filters = int(filters)
        self.kernel_size = kernel_size
        self.ratio = int(ratio)
        self.cheap_kernel_size = cheap_kernel_size
        self.strides = strides
        self.padding = padding
        self.groups = int(groups)
        self.activation_name = activation
        self.use_bias = use_bias

        self.kernel_initializer = tf.keras.initializers.get(kernel_initializer)
        self.bias_initializer = tf.keras.initializers.get(bias_initializer)
        self.activation_fn = (
            tf.keras.activations.get(activation)
            if activation is not None
            else None
        )

        self.input_channels = None
        self.primary_channels = None
        self.ghost_channels = None
        self.primary_conv = None
        self.cheap_conv = None

    def build(self, input_shape):
        self.input_channels = int(input_shape[-1])

        # Number of intrinsic feature maps.
        self.primary_channels = int(math.ceil(self.filters / self.ratio))
        self.primary_channels = max(1, self.primary_channels)

        # Number of cheap ghost feature maps needed.
        self.ghost_channels = self.filters - self.primary_channels

        primary_groups = _safe_conv_group_count(
            input_channels=self.input_channels,
            filters=self.primary_channels,
            requested_groups=self.groups
        )

        self.primary_conv = layers.Conv2D(
            filters=self.primary_channels,
            kernel_size=self.kernel_size,
            strides=self.strides,
            padding=self.padding,
            groups=primary_groups,
            use_bias=self.use_bias,
            kernel_initializer=self.kernel_initializer,
            bias_initializer=self.bias_initializer,
            name="primary_group_pointwise_conv"
        )

        if self.ghost_channels > 0:
            depth_multiplier = int(
                math.ceil(self.ghost_channels / self.primary_channels)
            )

            self.cheap_conv = layers.DepthwiseConv2D(
                kernel_size=self.cheap_kernel_size,
                strides=1,
                padding="same",
                depth_multiplier=depth_multiplier,
                use_bias=self.use_bias,
                depthwise_initializer=self.kernel_initializer,
                bias_initializer=self.bias_initializer,
                name="cheap_depthwise_ghost_conv"
            )

        super(GhostConv2D, self).build(input_shape)

    def call(self, inputs, training=None):
        primary = self.primary_conv(inputs)

        if self.activation_fn is not None:
            primary = self.activation_fn(primary)

        if self.cheap_conv is None:
            return primary[..., :self.filters]

        ghost = self.cheap_conv(primary)

        if self.activation_fn is not None:
            ghost = self.activation_fn(ghost)

        x = tf.concat([primary, ghost], axis=-1)

        # The cheap branch can produce slightly more channels than needed.
        return x[..., :self.filters]

    def get_config(self):
        config = super(GhostConv2D, self).get_config()
        config.update({
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "ratio": self.ratio,
            "cheap_kernel_size": self.cheap_kernel_size,
            "strides": self.strides,
            "padding": self.padding,
            "groups": self.groups,
            "activation": self.activation_name,
            "use_bias": self.use_bias,
            "kernel_initializer": tf.keras.initializers.serialize(self.kernel_initializer),
            "bias_initializer": tf.keras.initializers.serialize(self.bias_initializer),
        })
        return config


# ---------------------------------------------------------------------
# Low-rank dense replacement
# ---------------------------------------------------------------------

class LowRankDense(layers.Layer):
    """
    Explicit low-rank dense layer.

    Standard Dense:
        y = x @ W + b
        W shape = [input_dim, units]

    LowRankDense:
        W ≈ U @ V
        U shape = [input_dim, rank]
        V shape = [rank, units]

    Parameters are reduced when:
        rank * (input_dim + units) < input_dim * units
    """
    def __init__(
        self,
        units,
        rank=None,
        rank_ratio=0.50,
        min_rank=4,
        activation=None,
        use_bias=True,
        kernel_initializer="glorot_uniform",
        bias_initializer="zeros",
        **kwargs
    ):
        super(LowRankDense, self).__init__(**kwargs)

        if units < 1:
            raise ValueError("units must be >= 1.")
        if rank is not None and rank < 1:
            raise ValueError("rank must be None or >= 1.")
        if rank_ratio <= 0:
            raise ValueError("rank_ratio must be > 0.")

        self.units = int(units)
        self.rank = rank
        self.rank_ratio = float(rank_ratio)
        self.min_rank = int(min_rank)
        self.activation_name = activation
        self.use_bias = use_bias

        self.kernel_initializer = tf.keras.initializers.get(kernel_initializer)
        self.bias_initializer = tf.keras.initializers.get(bias_initializer)
        self.activation_fn = (
            tf.keras.activations.get(activation)
            if activation is not None
            else None
        )

        self.input_dim = None
        self.effective_rank = None
        self.kernel_left = None
        self.kernel_right = None
        self.bias = None

    def build(self, input_shape):
        self.input_dim = int(input_shape[-1])
        max_rank = min(self.input_dim, self.units)

        if self.rank is None:
            # Break-even rank for parameter reduction:
            # r * (input_dim + units) < input_dim * units
            break_even_rank = int(
                (self.input_dim * self.units) // (self.input_dim + self.units)
            )

            if break_even_rank <= 1:
                auto_rank = 1
            else:
                auto_rank = int(math.floor(break_even_rank * self.rank_ratio))
                auto_rank = max(1, auto_rank)
                auto_rank = max(self.min_rank, auto_rank)

                # Keep it below break-even when possible.
                auto_rank = min(auto_rank, break_even_rank - 1)

            self.effective_rank = min(max_rank, auto_rank)

        else:
            self.effective_rank = min(max_rank, int(self.rank))

        self.effective_rank = max(1, self.effective_rank)

        self.kernel_left = self.add_weight(
            name="kernel_left",
            shape=(self.input_dim, self.effective_rank),
            initializer=self.kernel_initializer,
            trainable=True
        )

        self.kernel_right = self.add_weight(
            name="kernel_right",
            shape=(self.effective_rank, self.units),
            initializer=self.kernel_initializer,
            trainable=True
        )

        if self.use_bias:
            self.bias = self.add_weight(
                name="bias",
                shape=(self.units,),
                initializer=self.bias_initializer,
                trainable=True
            )

        super(LowRankDense, self).build(input_shape)

    def call(self, inputs):
        x = tf.linalg.matmul(inputs, self.kernel_left)
        x = tf.linalg.matmul(x, self.kernel_right)

        if self.bias is not None:
            x = tf.nn.bias_add(x, self.bias)

        if self.activation_fn is not None:
            x = self.activation_fn(x)

        return x

    def get_config(self):
        config = super(LowRankDense, self).get_config()
        config.update({
            "units": self.units,
            "rank": self.rank,
            "rank_ratio": self.rank_ratio,
            "min_rank": self.min_rank,
            "activation": self.activation_name,
            "use_bias": self.use_bias,
            "kernel_initializer": tf.keras.initializers.serialize(self.kernel_initializer),
            "bias_initializer": tf.keras.initializers.serialize(self.bias_initializer),
        })
        return config

In [ ]:
import math
import tensorflow as tf
from tensorflow.keras import layers


# ---------------------------------------------------------------------
# Utility layers / functions
# ---------------------------------------------------------------------

def _safe_group_count(channels, requested_groups):
    """
    Returns a valid group count for Conv2D(groups=...).
    The returned value divides `channels`; if requested_groups is invalid,
    it falls back to gcd(channels, requested_groups), then to 1 if needed.
    """
    if channels is None:
        raise ValueError("The channel dimension must be statically known for grouped convolutions.")

    channels = int(channels)
    requested_groups = int(requested_groups)

    if channels <= 1 or requested_groups <= 1:
        return 1

    requested_groups = min(requested_groups, channels)

    if channels % requested_groups == 0:
        return requested_groups

    gcd_groups = math.gcd(channels, requested_groups)
    return max(1, gcd_groups)


def mila_intra_modal_fusion(a, b, c, axis=-1):
    """
    MILA intra-modal fusion:
        additive path    = a + b + c
        subtractive path = a - b - c
        re-aggregate     = concat(base, additive, subtractive)

    This avoids collapsing ((a+b+c)+(a-b-c)) into 2a in code,
    while retaining the additive and subtractive complementary paths
    described in the paper.
    """
    additive_path = a + b + c
    subtractive_path = a - b - c
    return tf.concat([a, additive_path, subtractive_path], axis=axis)


class GlobalMinPooling2D(layers.Layer):
    """
    Global minimum pooling for NHWC feature maps.
    """
    def __init__(self, keepdims=False, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)
        self.keepdims = keepdims

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2], keepdims=self.keepdims)

    def compute_output_shape(self, input_shape):
        if self.keepdims:
            return (input_shape[0], 1, 1, input_shape[-1])
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        config.update({"keepdims": self.keepdims})
        return config


class ChannelShuffle(layers.Layer):
    """
    Channel shuffle after multi-scale concatenation.
    """
    def __init__(self, groups=3, **kwargs):
        super(ChannelShuffle, self).__init__(**kwargs)
        self.requested_groups = groups
        self.effective_groups = None
        self.channels_per_group = None

    def build(self, input_shape):
        channels = input_shape[-1]
        self.effective_groups = _safe_group_count(channels, self.requested_groups)
        self.channels_per_group = int(channels) // self.effective_groups
        super(ChannelShuffle, self).build(input_shape)

    def call(self, inputs):
        shape = tf.shape(inputs)
        batch, height, width, channels = shape[0], shape[1], shape[2], shape[3]

        x = tf.reshape(
            inputs,
            [batch, height, width, self.effective_groups, self.channels_per_group]
        )
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [batch, height, width, channels])
        return x

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super(ChannelShuffle, self).get_config()
        config.update({"groups": self.requested_groups})
        return config


# ---------------------------------------------------------------------
# MILA homogeneous convolution basis
# ---------------------------------------------------------------------

class MILAHomogeneousBasis(layers.Layer):
    """
    Shared MILA homogeneous-basis template:

        B_Omega(x) = x + GPC(CS(concat({Omega_k(x)}_{k in K})))

    operator:
        "DWC" -> Multi-scale Depth-wise Homogeneous Convolutions  (MDHC)
        "GWC" -> Multi-scale Group-wise Homogeneous Convolutions  (MGHC)
        "DDC" -> Multi-scale Dilated Depth-wise Homogeneous Conv. (MDDHC)
    """
    def __init__(
        self,
        operator,
        kernels=(1, 3, 5),
        groups=8,
        dilation_rates=(1, 2, 3),
        activation=None,
        **kwargs
    ):
        super(MILAHomogeneousBasis, self).__init__(**kwargs)

        operator = operator.upper()
        if operator not in {"DWC", "GWC", "DDC"}:
            raise ValueError("operator must be one of {'DWC', 'GWC', 'DDC'}.")

        if len(kernels) != len(dilation_rates):
            raise ValueError("kernels and dilation_rates must have the same length.")

        self.operator = operator
        self.kernels = tuple(kernels)
        self.groups = groups
        self.dilation_rates = tuple(dilation_rates)
        self.activation = activation

        self.channels = None
        self.conv_branches = None
        self.channel_shuffle = None
        self.gpc = None
        self.act = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        group_count = _safe_group_count(self.channels, self.groups)

        self.conv_branches = []

        for idx, kernel_size in enumerate(self.kernels):
            if self.operator == "DWC":
                # Depth-wise convolution stream.
                branch = layers.DepthwiseConv2D(
                    kernel_size=kernel_size,
                    padding="same",
                    use_bias=False,
                    name=f"mdhc_dwc_k{kernel_size}"
                )

            elif self.operator == "GWC":
                # Group-wise convolution stream.
                branch = layers.Conv2D(
                    filters=self.channels,
                    kernel_size=kernel_size,
                    padding="same",
                    groups=group_count,
                    use_bias=False,
                    name=f"mghc_gwc_k{kernel_size}"
                )

            else:
                # Dilated depth-wise convolution stream.
                dilation_rate = self.dilation_rates[idx]
                branch = layers.DepthwiseConv2D(
                    kernel_size=kernel_size,
                    padding="same",
                    dilation_rate=dilation_rate,
                    use_bias=False,
                    name=f"mddhc_ddc_k{kernel_size}_d{dilation_rate}"
                )

            self.conv_branches.append(branch)

        # Since concat has len(kernels) * C channels, groups=len(kernels)
        # is always safe for channel shuffle.
        self.channel_shuffle = ChannelShuffle(
            groups=len(self.kernels),
            name=f"{self.operator.lower()}_channel_shuffle"
        )

        # Group point-wise convolution: 1x1 grouped projection back to C.
        '''self.gpc = layers.Conv2D(
            filters=self.channels,
            kernel_size=1,
            padding="same",
            groups=group_count,
            use_bias=False,
            name=f"{self.operator.lower()}_gpc"
        )'''

                # In MILAHomogeneousBasis.build(...)
        # Original: layers.Conv2D(... name=f"{self.operator.lower()}_gpc")
        self.gpc = GhostConv2D(
            filters=self.channels,
            kernel_size=1,
            ratio=2,
            cheap_kernel_size=3,
            padding="same",
            groups=group_count,
            use_bias=False,
            name=f"{self.operator.lower()}_gpc"
        )

        self.act = layers.Activation(self.activation) if self.activation is not None else None

        super(MILAHomogeneousBasis, self).build(input_shape)

    def call(self, inputs):
        multi_scale = [branch(inputs) for branch in self.conv_branches]
        x = tf.concat(multi_scale, axis=-1)
        x = self.channel_shuffle(x)
        x = self.gpc(x)

        if self.act is not None:
            x = self.act(x)

        # Residual connection preserves modality-specific information.
        return inputs + x

    def get_config(self):
        config = super(MILAHomogeneousBasis, self).get_config()
        config.update({
            "operator": self.operator,
            "kernels": self.kernels,
            "groups": self.groups,
            "dilation_rates": self.dilation_rates,
            "activation": self.activation,
        })
        return config


# ---------------------------------------------------------------------
# One MILA stage: MDHC -> MGHC/MDDHC -> intra-modal fusion
# ---------------------------------------------------------------------

class MILAStage(layers.Layer):
    """
    One p-th MILA stage.

    Phase I:
        base    = MDHC(h)
        grouped = MGHC(base)
        dilated = MDDHC(base)
        h_next  = f(I(base, grouped, dilated))

    The stage also returns the channel-context descriptor used in Phase II:
        d0      = GMP(base)
        d_plus  = GAP(base + grouped + dilated)
        d_minus = GMN(base - grouped - dilated)
        context = I(d0, d_plus, d_minus)
    """
    def __init__(
        self,
        kernels=(1, 3, 5),
        groups=8,
        dilation_rates=(1, 2, 3),
        feature_activation="relu",
        **kwargs
    ):
        super(MILAStage, self).__init__(**kwargs)

        self.kernels = tuple(kernels)
        self.groups = groups
        self.dilation_rates = tuple(dilation_rates)
        self.feature_activation = feature_activation

        self.mdhc = MILAHomogeneousBasis(
            operator="DWC",
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=(1, 1, 1),
            activation=None,
            name="mdhc"
        )

        self.mghc = MILAHomogeneousBasis(
            operator="GWC",
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=(1, 1, 1),
            activation=None,
            name="mghc"
        )

        self.mddhc = MILAHomogeneousBasis(
            operator="DDC",
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=self.dilation_rates,
            activation=None,
            name="mddhc"
        )

        self.channels = None
        self.fusion_projection = None
        self.fusion_activation = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        group_count = _safe_group_count(self.channels, self.groups)

        # Projects concat([base, additive, subtractive]) from 3C -> C.
        '''self.fusion_projection = layers.Conv2D(
            filters=self.channels,
            kernel_size=1,
            padding="same",
            groups=group_count,
            use_bias=False,
            name="mila_stage_fusion_gpc"
        )'''

                # In MILAStage.build(...)
        # Original: layers.Conv2D(... name="mila_stage_fusion_gpc")
        self.fusion_projection = GhostConv2D(
            filters=self.channels,
            kernel_size=1,
            ratio=2,
            cheap_kernel_size=3,
            padding="same",
            groups=group_count,
            use_bias=False,
            name="mila_stage_fusion_gpc"
        )

        self.fusion_activation = layers.Activation(self.feature_activation)

        super(MILAStage, self).build(input_shape)

    def call(self, inputs, training=None):
        # Phase I: multi-scale homogeneous streams.
        base = self.mdhc(inputs)
        grouped = self.mghc(base)
        dilated = self.mddhc(base)

        additive_path = base + grouped + dilated
        subtractive_path = base - grouped - dilated

        fused_feature = tf.concat([base, additive_path, subtractive_path], axis=-1)
        h_next = self.fusion_projection(fused_feature)
        h_next = self.fusion_activation(h_next)

        # Phase II descriptors for this stage.
        d0 = tf.reduce_max(base, axis=[1, 2])                  # GMP
        d_plus = tf.reduce_mean(additive_path, axis=[1, 2])    # GAP
        d_minus = tf.reduce_min(subtractive_path, axis=[1, 2]) # GMN

        stage_context = mila_intra_modal_fusion(d0, d_plus, d_minus, axis=-1)

        return h_next, stage_context

    def get_config(self):
        config = super(MILAStage, self).get_config()
        config.update({
            "kernels": self.kernels,
            "groups": self.groups,
            "dilation_rates": self.dilation_rates,
            "feature_activation": self.feature_activation,
        })
        return config


# ---------------------------------------------------------------------
# MILA channel attention map A_i^(E,j)
# ---------------------------------------------------------------------

class MILAChannelAttention(layers.Layer):
    """
    HySAR MILA Euclidean-space attention branch.

    Input:
        x: NHWC modality-specific contextual feature map.

    Output:
        A_E: MILA Euclidean channel attention map with shape [B, 1, 1, C].

    Default P=2, matching the paper's implementation.
    """
    def __init__(
        self,
        stages=2,
        reduction=4,
        min_units=8,
        kernels=(1, 3, 5),
        groups=8,
        dilation_rates=(1, 2, 3),
        feature_activation="relu",
        projection_activation="relu",
        use_scale=True,
        **kwargs
    ):
        super(MILAChannelAttention, self).__init__(**kwargs)

        if stages < 1:
            raise ValueError("stages must be >= 1.")

        self.stages = stages
        self.reduction = reduction
        self.min_units = min_units
        self.kernels = tuple(kernels)
        self.groups = groups
        self.dilation_rates = tuple(dilation_rates)
        self.feature_activation = feature_activation
        self.projection_activation = projection_activation
        self.use_scale = use_scale

        self.stage_layers = [
            MILAStage(
                kernels=self.kernels,
                groups=self.groups,
                dilation_rates=self.dilation_rates,
                feature_activation=self.feature_activation,
                name=f"mila_stage_{idx + 1}"
            )
            for idx in range(self.stages)
        ]

        self.channels = None
        self.low_rank_fc1 = None
        self.low_rank_fc2 = None
        self.logit_scale = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        bottleneck_units = max(self.channels // self.reduction, self.min_units)

        # Low-rank dense projection for multi-scale channel context.
        '''self.low_rank_fc1 = layers.Dense(
            units=bottleneck_units,
            activation=self.projection_activation,
            use_bias=True,
            name="mila_low_rank_fc1"
        )

        self.low_rank_fc2 = layers.Dense(
            units=self.channels,
            activation=None,
            use_bias=True,
            name="mila_low_rank_fc2"
        )'''

                # In MILAChannelAttention.build(...)
        # Original: layers.Dense(... name="mila_low_rank_fc1")
        self.low_rank_fc1 = LowRankDense(
            units=bottleneck_units,
            rank=None,          # auto-selects a compressed rank
            rank_ratio=0.50,    # lower = more compression; higher = more capacity
            min_rank=4,
            activation=self.projection_activation,
            use_bias=True,
            name="mila_low_rank_fc1"
        )
        
        # Original: layers.Dense(... name="mila_low_rank_fc2")
        self.low_rank_fc2 = LowRankDense(
            units=self.channels,
            rank=None,
            rank_ratio=0.50,
            min_rank=4,
            activation=None,
            use_bias=True,
            name="mila_low_rank_fc2"
        )

        if self.use_scale:
            # Scales logits before sigmoid so A_E remains in [0, 1].
            self.logit_scale = self.add_weight(
                name="mila_logit_scale",
                shape=(1, self.channels),
                initializer="ones",
                trainable=True
            )

        super(MILAChannelAttention, self).build(input_shape)

    def call(self, inputs, training=None):
        h = inputs
        stage_contexts = []

        # Phase I across p-th MILA stages.
        for stage in self.stage_layers:
            h, context = stage(h, training=training)
            stage_contexts.append(context)

        # Phase II: fuse descriptors across stages.
        # Each context has shape [B, 3C].
        multi_stage_context = tf.add_n(stage_contexts)

        logits = self.low_rank_fc1(multi_stage_context)
        logits = self.low_rank_fc2(logits)

        if self.logit_scale is not None:
            logits = logits * self.logit_scale

        attention = tf.nn.sigmoid(logits)
        attention = tf.reshape(
            attention,
            [tf.shape(inputs)[0], 1, 1, self.channels]
        )

        return attention

    def get_config(self):
        config = super(MILAChannelAttention, self).get_config()
        config.update({
            "stages": self.stages,
            "reduction": self.reduction,
            "min_units": self.min_units,
            "kernels": self.kernels,
            "groups": self.groups,
            "dilation_rates": self.dilation_rates,
            "feature_activation": self.feature_activation,
            "projection_activation": self.projection_activation,
            "use_scale": self.use_scale,
        })
        return config


class MILAFeatureRefiner(layers.Layer):
    """
    Single-modality MILA feature refiner.

    Returns:
        x_refined = x * A_E * alpha

    Use this when you want MILA alone for one modality.
    """
    def __init__(
        self,
        stages=2,
        reduction=4,
        min_units=8,
        kernels=(1, 3, 5),
        groups=8,
        dilation_rates=(1, 2, 3),
        feature_activation="relu",
        projection_activation="relu",
        use_scale=True,
        **kwargs
    ):
        super(MILAFeatureRefiner, self).__init__(**kwargs)

        self.stages = stages
        self.reduction = reduction
        self.min_units = min_units
        self.kernels = tuple(kernels)
        self.groups = groups
        self.dilation_rates = tuple(dilation_rates)
        self.feature_activation = feature_activation
        self.projection_activation = projection_activation
        self.use_scale = use_scale

        self.mila_attention = MILAChannelAttention(
            stages=self.stages,
            reduction=self.reduction,
            min_units=self.min_units,
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=self.dilation_rates,
            feature_activation=self.feature_activation,
            projection_activation=self.projection_activation,
            use_scale=self.use_scale,
            name="mila_attention"
        )

        self.alpha = None

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.alpha = self.add_weight(
            name="mila_feature_alpha",
            shape=(1, 1, 1, channels),
            initializer="ones",
            trainable=True
        )
        super(MILAFeatureRefiner, self).build(input_shape)

    def call(self, inputs, training=None):
        attention = self.mila_attention(inputs, training=training)
        return inputs * attention * self.alpha

    def get_config(self):
        config = super(MILAFeatureRefiner, self).get_config()
        config.update({
            "stages": self.stages,
            "reduction": self.reduction,
            "min_units": self.min_units,
            "kernels": self.kernels,
            "groups": self.groups,
            "dilation_rates": self.dilation_rates,
            "feature_activation": self.feature_activation,
            "projection_activation": self.projection_activation,
            "use_scale": self.use_scale,
        })
        return config


# ---------------------------------------------------------------------
# Backward-compatible replacement for your attached two-input wrapper
# ---------------------------------------------------------------------

class DeeperGlobalLocalAttentionLayer1(layers.Layer):
    """
    Backward-compatible name for your old EMCAM attention-map class.

    Important:
        Your original class returned one shared attention map for two inputs.
        MILA is modality-specific, so this revised class returns:
            A_E_1, A_E_2

        where each A_E_i has shape [B, 1, 1, C_i].
    """
    def __init__(
        self,
        units=None,
        activation="relu",
        dropout_rate=0.0,
        groups=8,
        use_scale=True,
        stages=2,
        reduction=4,
        min_units=8,
        kernels=(1, 3, 5),
        dilation_rates=(1, 2, 3),
        **kwargs
    ):
        super(DeeperGlobalLocalAttentionLayer1, self).__init__(**kwargs)

        # Kept only for compatibility with your previous constructor.
        self.units = units
        self.activation = activation
        self.dropout_rate = dropout_rate

        self.groups = groups
        self.use_scale = use_scale
        self.stages = stages
        self.reduction = reduction
        self.min_units = min_units
        self.kernels = tuple(kernels)
        self.dilation_rates = tuple(dilation_rates)

        self.mila_attention1 = MILAChannelAttention(
            stages=self.stages,
            reduction=self.reduction,
            min_units=self.min_units,
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=self.dilation_rates,
            feature_activation=self.activation,
            projection_activation=self.activation,
            use_scale=self.use_scale,
            name="mila_attention_modality_1"
        )

        self.mila_attention2 = MILAChannelAttention(
            stages=self.stages,
            reduction=self.reduction,
            min_units=self.min_units,
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=self.dilation_rates,
            feature_activation=self.activation,
            projection_activation=self.activation,
            use_scale=self.use_scale,
            name="mila_attention_modality_2"
        )

    def call(self, inputs, training=None):
        input1, input2 = inputs

        attention1 = self.mila_attention1(input1, training=training)
        attention2 = self.mila_attention2(input2, training=training)

        return attention1, attention2

    def get_config(self):
        config = super(DeeperGlobalLocalAttentionLayer1, self).get_config()
        config.update({
            "units": self.units,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
            "groups": self.groups,
            "use_scale": self.use_scale,
            "stages": self.stages,
            "reduction": self.reduction,
            "min_units": self.min_units,
            "kernels": self.kernels,
            "dilation_rates": self.dilation_rates,
        })
        return config


class DeeperAttentionLayer1(layers.Layer):
    """
    Drop-in replacement for your old two-modality EMCAM integration wrapper.

    Same call pattern as your attached code:
        attention_feature1, attention_feature2 = layer([input1, input2])

    Revised MILA behavior:
        A_E_1 = MILA(input1)
        A_E_2 = MILA(input2)

        output1 = input1 * A_E_1 * alpha3
        output2 = input2 * A_E_2 * alpha4
    """
    def __init__(
        self,
        units=64,
        use_scale=True,
        stages=2,
        reduction=4,
        min_units=8,
        groups=8,
        kernels=(1, 3, 5),
        dilation_rates=(1, 2, 3),
        activation="relu",
        **kwargs
    ):
        super(DeeperAttentionLayer1, self).__init__(**kwargs)

        self.units = units
        self.use_scale = use_scale
        self.stages = stages
        self.reduction = reduction
        self.min_units = min_units
        self.groups = groups
        self.kernels = tuple(kernels)
        self.dilation_rates = tuple(dilation_rates)
        self.activation = activation

        self.deeper_global_local_attention = DeeperGlobalLocalAttentionLayer1(
            units=self.units,
            activation=self.activation,
            dropout_rate=0.0,
            groups=self.groups,
            use_scale=self.use_scale,
            stages=self.stages,
            reduction=self.reduction,
            min_units=self.min_units,
            kernels=self.kernels,
            dilation_rates=self.dilation_rates,
            name="hysar_mila_attention_maps"
        )

        self.alpha3 = None
        self.alpha4 = None

    def build(self, input_shape):
        input1_shape, input2_shape = input_shape
        channels1 = int(input1_shape[-1])
        channels2 = int(input2_shape[-1])

        self.alpha3 = self.add_weight(
            name="alpha3_mila_modality_1",
            shape=(1, 1, 1, channels1),
            initializer="ones",
            trainable=True
        )

        self.alpha4 = self.add_weight(
            name="alpha4_mila_modality_2",
            shape=(1, 1, 1, channels2),
            initializer="ones",
            trainable=True
        )

        super(DeeperAttentionLayer1, self).build(input_shape)

    def call(self, inputs, training=None):
        input1, input2 = inputs

        attention1, attention2 = self.deeper_global_local_attention(
            [input1, input2],
            training=training
        )

        attention_feature1 = input1 * attention1 * self.alpha3
        attention_feature2 = input2 * attention2 * self.alpha4

        return attention_feature1, attention_feature2

    def get_config(self):
        config = super(DeeperAttentionLayer1, self).get_config()
        config.update({
            "units": self.units,
            "use_scale": self.use_scale,
            "stages": self.stages,
            "reduction": self.reduction,
            "min_units": self.min_units,
            "groups": self.groups,
            "kernels": self.kernels,
            "dilation_rates": self.dilation_rates,
            "activation": self.activation,
        })
        return config


# ---------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------
#
# input1: WSI / modality-1 feature map, shape [B, H, W, C1]
# input2: omics-as-feature-map / modality-2 feature map, shape [B, H, W, C2]
#
# mila_layer = DeeperAttentionLayer1(
#     units=64,        # kept for compatibility; MILA internally uses input channels
#     stages=2,        # P=2
#     reduction=4,
#     groups=8,
#     use_scale=True
# )
#
# refined_input1, refined_input2 = mila_layer([input1, input2])
#
# For single-modality use:
#
# single_mila = MILAFeatureRefiner(stages=2, reduction=4, groups=8)
# refined_x = single_mila(x)
#
# For full HySAR later:
#     A_E = MILAChannelAttention(...)(x)
# and pass A_E into your MCF together with HyGAM's non-Euclidean map.

In [ ]:
## MSTL
def residual_GLC_branch1(inputs1, inputs2):
    
    x1 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs1)
    #x1 = DeeperAttentionLayer1(units=64, use_scale=True)(x1) ## MFA ####
    #x1 = AttentionBlock(64)(x1)
    x1 = BatchNormalization()(x1)
    x1 = tf.keras.layers.Activation('relu')(x1)
    x1 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x1)
    
    x2 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs2)
    #x2 = DeeperAttentionLayer1(units=64, use_scale=True)(x2) ## MFA ####
    #x2 = AttentionBlock(64)(x2)
    x2 = BatchNormalization()(x2)
    x2 = tf.keras.layers.Activation('relu')(x2)
    x2 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x2)
    
    ## ERLA
    x1 = RGSA(x1, filters=64)
    #
    x2 = RGSA(x2, filters=64)
    
    
    
    
    x1 = RGSA(x1, filters=64)
    
    x2 = RGSA(x2, filters=64)
    
    #x1, x2 = DeeperAttentionLayer1(units=64, use_scale=True)([x1, x2])

    '''mila_layer = DeeperAttentionLayer1(units=64, stages=2, reduction=4, groups=8, use_scale=True)
    x1, x2 = mila_layer([x1, x2])
    '''
    
    ## ERLA
    x1 = RGSA(x1, filters=128, strides=(2, 2), use_projection=True)
    
    x2 = RGSA(x2, filters=128, strides=(2, 2), use_projection=True)
    
    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    
    x1 = RGSA(x1, filters=128)
    
    x2 = RGSA(x2, filters=128)

    
    #x1, x2 = DeeperAttentionLayer1(units=128, use_scale=True)([x1, x2])

    """mila_layer = DeeperAttentionLayer1(units=128, stages=2, reduction=4, groups=8, use_scale=True)
    x1, x2 = mila_layer([x1, x2])
    """
    
    ## ERLA
    x1 = RGSA(x1, filters=256, strides=(2, 2), use_projection=True)
    
    
    x2 = RGSA(x2, filters=256, strides=(2, 2), use_projection=True)
    

    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    
    x1 = RGSA(x1, filters=256)
    
    x2 = RGSA(x2, filters=256)
    
    #x1, x2 = DeeperAttentionLayer1(units=256, use_scale=True)([x1, x2])

    '''mila_layer = DeeperAttentionLayer1(units=256, stages=2, reduction=8, groups=16, use_scale=True)
    x1, x2 = mila_layer([x1, x2])
    '''
    
    
    ## ERLA
    x1 = RGSA(x1, filters=512, strides=(2, 2), use_projection=True)
    
    x2 = RGSA(x2, filters=512, strides=(2, 2), use_projection=True)

    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    ## ERLA
    x1 = RGSA(x1, filters=512)
    x2 = RGSA(x2, filters=512)

    #x1, x2 = DeeperAttentionLayer1(units=512, use_scale=True)([x1, x2])

    '''mila_layer = DeeperAttentionLayer1(units=512, stages=2, reduction=8, groups=16, use_scale=True)
    x1, x2 = mila_layer([x1, x2])'''

    
    
    return x1, x2

In [ ]:
#def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
input_shape=(128, 128, 3)
inputs1 = Input(shape=input_shape)
inputs2 = Input(shape=input_shape)

import tensorflow.keras.layers as L

#input_data = Input(shape=input_shape, name='input_data')
# Initial convolutional layer

x1, x2 = residual_GLC_branch1(inputs1, inputs2)
#print('x:',x.shape)


con = tf.keras.layers.Concatenate(axis=-1)([x1, x2])

#con = tf.keras.layers.Dropout(0.25)(con) 

x = GlobalAveragePooling2D()(con)
#print('GlobalAveragePooling2D x:',x.shape)

## TMTL
outputs1 = Dense(5, activation='softmax')(x)
outputs2 = Dense(7, activation='softmax')(x)

# Create the model
model = Model([inputs1, inputs2], [outputs1, outputs2])
#return model
print(model.summary())

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Add, Concatenate

def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(
        [tf.TensorSpec([1, *inputs.shape[1:]]) for inputs in model.inputs]
    )
    
    # Calculate FLOPs using TensorFlow's built-in profiler
    graph = concrete_func.graph
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    
    flops = tf.compat.v1.profiler.profile(
        graph,
        run_meta=run_meta,
        cmd='op',
        options=opts
    )
    
    return flops.total_float_ops

# Load your trained model or build it using the provided code
# model = ... (your model loading/building code here)

# Calculate total FLOPs
total_flops = get_flops(model)
gflops = total_flops / 1e9

print(f"Total FLOPs: {total_flops:,}")
print(f"GFLOPs: {gflops:.2f}")

## HySAR ##

In [ ]:
import math
import tensorflow as tf
from tensorflow.keras import layers


# ============================================================
# Utility functions
# ============================================================

def _safe_group_count(channels, requested_groups):
    """
    Safe group count for grouped/depthwise-related operations.
    Returned group count always divides channels.
    """
    if channels is None:
        raise ValueError("Channel dimension must be statically known.")

    channels = int(channels)
    requested_groups = int(requested_groups)

    if channels <= 1 or requested_groups <= 1:
        return 1

    requested_groups = min(requested_groups, channels)

    if channels % requested_groups == 0:
        return requested_groups

    return max(1, math.gcd(channels, requested_groups))


def _safe_conv_group_count(input_channels, filters, requested_groups):
    """
    Safe group count for Conv2D(groups=g).

    Keras requires:
        input_channels % g == 0
        filters % g == 0
    """
    if input_channels is None or filters is None:
        raise ValueError("Input channels and filters must be statically known.")

    input_channels = int(input_channels)
    filters = int(filters)
    requested_groups = int(requested_groups)

    if requested_groups <= 1:
        return 1

    requested_groups = min(requested_groups, input_channels, filters)

    return max(
        1,
        math.gcd(math.gcd(input_channels, filters), requested_groups)
    )


def _znorm_vector(v, eps=1e-6):
    """
    Per-sample vector normalization for stable HyGAM geometry logits.
    """
    mean = tf.reduce_mean(v, axis=-1, keepdims=True)
    std = tf.math.reduce_std(v, axis=-1, keepdims=True) + eps
    return (v - mean) / std


def _rank_for(in_dim, out_dim, reduction=4, min_rank=4):
    """
    Practical low-rank size helper.
    """
    in_dim = int(in_dim)
    out_dim = int(out_dim)
    reduction = int(reduction)

    max_rank = max(1, min(in_dim, out_dim))
    rank = max(1, max_rank // max(1, reduction))
    rank = max(rank, int(min_rank))
    return min(rank, max_rank)


# ============================================================
# Global minimum pooling
# ============================================================

class GlobalMinPooling2D(layers.Layer):
    def __init__(self, keepdims=False, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)
        self.keepdims = keepdims

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2], keepdims=self.keepdims)

    def compute_output_shape(self, input_shape):
        if self.keepdims:
            return (input_shape[0], 1, 1, input_shape[-1])
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        config.update({"keepdims": self.keepdims})
        return config


# ============================================================
# Low-rank dense layer
# ============================================================

class LowRankDense(layers.Layer):
    """
    Low-rank replacement for Dense.

    Dense:
        y = xW + b

    LowRankDense:
        W ≈ U @ V
        y = xU V + b

    This reduces parameters when rank is small.
    """
    def __init__(
        self,
        units,
        rank=None,
        rank_ratio=0.50,
        min_rank=4,
        activation=None,
        use_bias=True,
        kernel_initializer="glorot_uniform",
        bias_initializer="zeros",
        **kwargs
    ):
        super(LowRankDense, self).__init__(**kwargs)

        self.units = int(units)
        self.rank = rank
        self.rank_ratio = float(rank_ratio)
        self.min_rank = int(min_rank)
        self.activation_name = activation
        self.use_bias = bool(use_bias)

        self.kernel_initializer = tf.keras.initializers.get(kernel_initializer)
        self.bias_initializer = tf.keras.initializers.get(bias_initializer)

        self.activation_fn = (
            tf.keras.activations.get(activation)
            if activation is not None
            else None
        )

        self.input_dim = None
        self.effective_rank = None
        self.kernel_left = None
        self.kernel_right = None
        self.bias = None

    def build(self, input_shape):
        self.input_dim = int(input_shape[-1])
        max_rank = max(1, min(self.input_dim, self.units))

        if self.rank is None:
            break_even_rank = int(
                (self.input_dim * self.units) // max(1, self.input_dim + self.units)
            )

            if break_even_rank <= 1:
                auto_rank = 1
            else:
                auto_rank = int(math.floor(break_even_rank * self.rank_ratio))
                auto_rank = max(1, auto_rank)
                auto_rank = max(self.min_rank, auto_rank)
                auto_rank = min(auto_rank, break_even_rank - 1)

            self.effective_rank = min(max_rank, auto_rank)
        else:
            self.effective_rank = min(max_rank, int(self.rank))

        self.effective_rank = max(1, self.effective_rank)

        self.kernel_left = self.add_weight(
            name="kernel_left",
            shape=(self.input_dim, self.effective_rank),
            initializer=self.kernel_initializer,
            trainable=True
        )

        self.kernel_right = self.add_weight(
            name="kernel_right",
            shape=(self.effective_rank, self.units),
            initializer=self.kernel_initializer,
            trainable=True
        )

        if self.use_bias:
            self.bias = self.add_weight(
                name="bias",
                shape=(self.units,),
                initializer=self.bias_initializer,
                trainable=True
            )

        super(LowRankDense, self).build(input_shape)

    def call(self, inputs):
        x = tf.tensordot(inputs, self.kernel_left, axes=[[-1], [0]])
        x = tf.tensordot(x, self.kernel_right, axes=[[-1], [0]])

        if self.bias is not None:
            x = x + self.bias

        if self.activation_fn is not None:
            x = self.activation_fn(x)

        return x

    def get_config(self):
        config = super(LowRankDense, self).get_config()
        config.update({
            "units": self.units,
            "rank": self.rank,
            "rank_ratio": self.rank_ratio,
            "min_rank": self.min_rank,
            "activation": self.activation_name,
            "use_bias": self.use_bias,
            "kernel_initializer": tf.keras.initializers.serialize(self.kernel_initializer),
            "bias_initializer": tf.keras.initializers.serialize(self.bias_initializer),
        })
        return config


# ============================================================
# Ghost convolution for MILA GPC / fusion projection
# ============================================================

class GhostConv2D(layers.Layer):
    """
    GhostConv2D replacement for expensive 1x1/group point-wise convolution.

    For MILA:
        kernel_size=1
        groups=group_count
        ratio=2
    """
    def __init__(
        self,
        filters,
        kernel_size=1,
        ratio=2,
        cheap_kernel_size=3,
        strides=1,
        padding="same",
        groups=1,
        activation=None,
        use_bias=False,
        kernel_initializer="glorot_uniform",
        bias_initializer="zeros",
        **kwargs
    ):
        super(GhostConv2D, self).__init__(**kwargs)

        self.filters = int(filters)
        self.kernel_size = kernel_size
        self.ratio = int(ratio)
        self.cheap_kernel_size = cheap_kernel_size
        self.strides = strides
        self.padding = padding
        self.groups = int(groups)
        self.activation_name = activation
        self.use_bias = bool(use_bias)

        self.kernel_initializer = tf.keras.initializers.get(kernel_initializer)
        self.bias_initializer = tf.keras.initializers.get(bias_initializer)

        self.activation_fn = (
            tf.keras.activations.get(activation)
            if activation is not None
            else None
        )

        self.input_channels = None
        self.primary_channels = None
        self.ghost_channels = None

        self.primary_conv = None
        self.cheap_conv = None

    def build(self, input_shape):
        self.input_channels = int(input_shape[-1])

        self.primary_channels = int(math.ceil(self.filters / self.ratio))
        self.primary_channels = max(1, self.primary_channels)

        self.ghost_channels = self.filters - self.primary_channels

        primary_groups = _safe_conv_group_count(
            input_channels=self.input_channels,
            filters=self.primary_channels,
            requested_groups=self.groups
        )

        self.primary_conv = layers.Conv2D(
            filters=self.primary_channels,
            kernel_size=self.kernel_size,
            strides=self.strides,
            padding=self.padding,
            groups=primary_groups,
            use_bias=self.use_bias,
            kernel_initializer=self.kernel_initializer,
            bias_initializer=self.bias_initializer,
            name="primary_group_pointwise_conv"
        )

        if self.ghost_channels > 0:
            depth_multiplier = int(
                math.ceil(self.ghost_channels / self.primary_channels)
            )

            self.cheap_conv = layers.DepthwiseConv2D(
                kernel_size=self.cheap_kernel_size,
                strides=1,
                padding="same",
                depth_multiplier=depth_multiplier,
                use_bias=self.use_bias,
                depthwise_initializer=self.kernel_initializer,
                bias_initializer=self.bias_initializer,
                name="cheap_depthwise_ghost_conv"
            )

        super(GhostConv2D, self).build(input_shape)

    def call(self, inputs, training=None):
        primary = self.primary_conv(inputs)

        if self.activation_fn is not None:
            primary = self.activation_fn(primary)

        if self.cheap_conv is None:
            return primary[..., :self.filters]

        ghost = self.cheap_conv(primary)

        if self.activation_fn is not None:
            ghost = self.activation_fn(ghost)

        x = tf.concat([primary, ghost], axis=-1)
        return x[..., :self.filters]

    def get_config(self):
        config = super(GhostConv2D, self).get_config()
        config.update({
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "ratio": self.ratio,
            "cheap_kernel_size": self.cheap_kernel_size,
            "strides": self.strides,
            "padding": self.padding,
            "groups": self.groups,
            "activation": self.activation_name,
            "use_bias": self.use_bias,
            "kernel_initializer": tf.keras.initializers.serialize(self.kernel_initializer),
            "bias_initializer": tf.keras.initializers.serialize(self.bias_initializer),
        })
        return config


# ============================================================
# Channel shuffle
# ============================================================

class ChannelShuffle(layers.Layer):
    def __init__(self, groups=3, **kwargs):
        super(ChannelShuffle, self).__init__(**kwargs)
        self.requested_groups = int(groups)
        self.effective_groups = None
        self.channels_per_group = None

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.effective_groups = _safe_group_count(channels, self.requested_groups)
        self.channels_per_group = channels // self.effective_groups
        super(ChannelShuffle, self).build(input_shape)

    def call(self, inputs):
        shape = tf.shape(inputs)
        batch = shape[0]
        height = shape[1]
        width = shape[2]
        channels = shape[3]

        x = tf.reshape(
            inputs,
            [batch, height, width, self.effective_groups, self.channels_per_group]
        )
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [batch, height, width, channels])
        return x

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super(ChannelShuffle, self).get_config()
        config.update({"groups": self.requested_groups})
        return config


# ============================================================
# MILA: Euclidean branch
# ============================================================

def mila_intra_modal_fusion(a, b, c, axis=-1):
    """
    MILA intra-modal fusion.

    It preserves:
        base path        = a
        additive path    = a + b + c
        subtractive path = a - b - c

    Then it concatenates them.
    """
    additive_path = a + b + c
    subtractive_path = a - b - c
    return tf.concat([a, additive_path, subtractive_path], axis=axis)


class MILAHomogeneousBasis(layers.Layer):
    """
    Shared MILA homogeneous convolution basis.

    B_Omega(x) =
        x + GhostGPC(ChannelShuffle(Concat({Omega_k(x)})))

    operator:
        DWC -> multi-scale depth-wise homogeneous convolution
        GWC -> multi-scale group-wise homogeneous convolution
        DDC -> multi-scale dilated depth-wise homogeneous convolution
    """
    def __init__(
        self,
        operator,
        kernels=(1, 3, 5),
        groups=8,
        dilation_rates=(1, 2, 3),
        ghost_ratio=2,
        ghost_kernel_size=3,
        activation=None,
        **kwargs
    ):
        super(MILAHomogeneousBasis, self).__init__(**kwargs)

        operator = operator.upper()
        if operator not in {"DWC", "GWC", "DDC"}:
            raise ValueError("operator must be one of {'DWC', 'GWC', 'DDC'}.")

        self.operator = operator
        self.kernels = tuple(kernels)
        self.groups = int(groups)
        self.dilation_rates = tuple(dilation_rates)
        self.ghost_ratio = int(ghost_ratio)
        self.ghost_kernel_size = int(ghost_kernel_size)
        self.activation = activation

        self.channels = None
        self.conv_branches = None
        self.channel_shuffle = None
        self.gpc = None
        self.act = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        group_count = _safe_group_count(self.channels, self.groups)

        self.conv_branches = []

        for idx, kernel_size in enumerate(self.kernels):
            if self.operator == "DWC":
                branch = layers.DepthwiseConv2D(
                    kernel_size=kernel_size,
                    padding="same",
                    use_bias=False,
                    name=f"mdhc_dwc_k{kernel_size}"
                )

            elif self.operator == "GWC":
                branch = layers.Conv2D(
                    filters=self.channels,
                    kernel_size=kernel_size,
                    padding="same",
                    groups=group_count,
                    use_bias=False,
                    name=f"mghc_gwc_k{kernel_size}"
                )

            else:
                dilation_rate = self.dilation_rates[idx]
                branch = layers.DepthwiseConv2D(
                    kernel_size=kernel_size,
                    padding="same",
                    dilation_rate=dilation_rate,
                    use_bias=False,
                    name=f"mddhc_ddc_k{kernel_size}_d{dilation_rate}"
                )

            self.conv_branches.append(branch)

        self.channel_shuffle = ChannelShuffle(
            groups=len(self.kernels),
            name=f"{self.operator.lower()}_channel_shuffle"
        )

        self.gpc = GhostConv2D(
            filters=self.channels,
            kernel_size=1,
            ratio=self.ghost_ratio,
            cheap_kernel_size=self.ghost_kernel_size,
            padding="same",
            groups=group_count,
            use_bias=False,
            name=f"{self.operator.lower()}_ghost_gpc"
        )

        self.act = (
            layers.Activation(self.activation)
            if self.activation is not None
            else None
        )

        super(MILAHomogeneousBasis, self).build(input_shape)

    def call(self, inputs, training=None):
        multi_scale = [branch(inputs) for branch in self.conv_branches]
        x = tf.concat(multi_scale, axis=-1)
        x = self.channel_shuffle(x)
        x = self.gpc(x, training=training)

        if self.act is not None:
            x = self.act(x)

        return inputs + x

    def get_config(self):
        config = super(MILAHomogeneousBasis, self).get_config()
        config.update({
            "operator": self.operator,
            "kernels": self.kernels,
            "groups": self.groups,
            "dilation_rates": self.dilation_rates,
            "ghost_ratio": self.ghost_ratio,
            "ghost_kernel_size": self.ghost_kernel_size,
            "activation": self.activation,
        })
        return config


class MILAStage(layers.Layer):
    """
    One MILA stage:

        base    = MDHC(h)
        grouped = MGHC(base)
        dilated = MDDHC(base)

        h_next = f( projection(concat(base,
                                      base + grouped + dilated,
                                      base - grouped - dilated)) )

    Also returns channel context using GMP, GAP, and GMN.
    """
    def __init__(
        self,
        kernels=(1, 3, 5),
        groups=8,
        dilation_rates=(1, 2, 3),
        ghost_ratio=2,
        ghost_kernel_size=3,
        feature_activation="relu",
        **kwargs
    ):
        super(MILAStage, self).__init__(**kwargs)

        self.kernels = tuple(kernels)
        self.groups = int(groups)
        self.dilation_rates = tuple(dilation_rates)
        self.ghost_ratio = int(ghost_ratio)
        self.ghost_kernel_size = int(ghost_kernel_size)
        self.feature_activation = feature_activation

        self.mdhc = MILAHomogeneousBasis(
            operator="DWC",
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=(1, 1, 1),
            ghost_ratio=self.ghost_ratio,
            ghost_kernel_size=self.ghost_kernel_size,
            activation=None,
            name="mdhc"
        )

        self.mghc = MILAHomogeneousBasis(
            operator="GWC",
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=(1, 1, 1),
            ghost_ratio=self.ghost_ratio,
            ghost_kernel_size=self.ghost_kernel_size,
            activation=None,
            name="mghc"
        )

        self.mddhc = MILAHomogeneousBasis(
            operator="DDC",
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=self.dilation_rates,
            ghost_ratio=self.ghost_ratio,
            ghost_kernel_size=self.ghost_kernel_size,
            activation=None,
            name="mddhc"
        )

        self.channels = None
        self.fusion_projection = None
        self.fusion_activation = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        group_count = _safe_group_count(self.channels, self.groups)

        self.fusion_projection = GhostConv2D(
            filters=self.channels,
            kernel_size=1,
            ratio=self.ghost_ratio,
            cheap_kernel_size=self.ghost_kernel_size,
            padding="same",
            groups=group_count,
            use_bias=False,
            name="mila_stage_fusion_ghost_gpc"
        )

        self.fusion_activation = layers.Activation(self.feature_activation)

        super(MILAStage, self).build(input_shape)

    def call(self, inputs, training=None):
        base = self.mdhc(inputs, training=training)
        grouped = self.mghc(base, training=training)
        dilated = self.mddhc(base, training=training)

        additive_path = base + grouped + dilated
        subtractive_path = base - grouped - dilated

        fused_feature = tf.concat([base, additive_path, subtractive_path], axis=-1)

        h_next = self.fusion_projection(fused_feature, training=training)
        h_next = self.fusion_activation(h_next)

        d0 = tf.reduce_max(base, axis=[1, 2])
        d_plus = tf.reduce_mean(additive_path, axis=[1, 2])
        d_minus = tf.reduce_min(subtractive_path, axis=[1, 2])

        stage_context = mila_intra_modal_fusion(
            d0,
            d_plus,
            d_minus,
            axis=-1
        )

        return h_next, stage_context

    def get_config(self):
        config = super(MILAStage, self).get_config()
        config.update({
            "kernels": self.kernels,
            "groups": self.groups,
            "dilation_rates": self.dilation_rates,
            "ghost_ratio": self.ghost_ratio,
            "ghost_kernel_size": self.ghost_kernel_size,
            "feature_activation": self.feature_activation,
        })
        return config


class MILAChannelAttention(layers.Layer):
    """
    MILA Euclidean-space channel attention.

    Output:
        A_E with shape [B, 1, 1, C]
    """
    def __init__(
        self,
        stages=2,
        reduction=8,
        min_units=8,
        kernels=(1, 3, 5),
        groups=8,
        dilation_rates=(1, 2, 3),
        ghost_ratio=2,
        ghost_kernel_size=3,
        feature_activation="relu",
        projection_activation="relu",
        use_scale=True,
        **kwargs
    ):
        super(MILAChannelAttention, self).__init__(**kwargs)

        self.stages = int(stages)
        self.reduction = int(reduction)
        self.min_units = int(min_units)
        self.kernels = tuple(kernels)
        self.groups = int(groups)
        self.dilation_rates = tuple(dilation_rates)
        self.ghost_ratio = int(ghost_ratio)
        self.ghost_kernel_size = int(ghost_kernel_size)
        self.feature_activation = feature_activation
        self.projection_activation = projection_activation
        self.use_scale = bool(use_scale)

        self.stage_layers = [
            MILAStage(
                kernels=self.kernels,
                groups=self.groups,
                dilation_rates=self.dilation_rates,
                ghost_ratio=self.ghost_ratio,
                ghost_kernel_size=self.ghost_kernel_size,
                feature_activation=self.feature_activation,
                name=f"mila_stage_{idx + 1}"
            )
            for idx in range(self.stages)
        ]

        self.channels = None
        self.low_rank_fc1 = None
        self.low_rank_fc2 = None
        self.logit_scale = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        bottleneck_units = max(self.channels // self.reduction, self.min_units)

        self.low_rank_fc1 = LowRankDense(
            units=bottleneck_units,
            rank=None,
            rank_ratio=0.50,
            min_rank=4,
            activation=self.projection_activation,
            use_bias=True,
            name="mila_low_rank_fc1"
        )

        self.low_rank_fc2 = LowRankDense(
            units=self.channels,
            rank=None,
            rank_ratio=0.50,
            min_rank=4,
            activation=None,
            use_bias=True,
            name="mila_low_rank_fc2"
        )

        if self.use_scale:
            self.logit_scale = self.add_weight(
                name="mila_logit_scale",
                shape=(1, self.channels),
                initializer="ones",
                trainable=True
            )

        super(MILAChannelAttention, self).build(input_shape)

    def call(self, inputs, training=None):
        h = inputs
        stage_contexts = []

        for stage in self.stage_layers:
            h, context = stage(h, training=training)
            stage_contexts.append(context)

        multi_stage_context = tf.add_n(stage_contexts)

        logits = self.low_rank_fc1(multi_stage_context)
        logits = self.low_rank_fc2(logits)

        if self.logit_scale is not None:
            logits = logits * self.logit_scale

        attention = tf.nn.sigmoid(logits)
        attention = tf.reshape(
            attention,
            [tf.shape(inputs)[0], 1, 1, self.channels]
        )

        return attention

    def get_config(self):
        config = super(MILAChannelAttention, self).get_config()
        config.update({
            "stages": self.stages,
            "reduction": self.reduction,
            "min_units": self.min_units,
            "kernels": self.kernels,
            "groups": self.groups,
            "dilation_rates": self.dilation_rates,
            "ghost_ratio": self.ghost_ratio,
            "ghost_kernel_size": self.ghost_kernel_size,
            "feature_activation": self.feature_activation,
            "projection_activation": self.projection_activation,
            "use_scale": self.use_scale,
        })
        return config


# ============================================================
# HyGAM: non-Euclidean hyperbolic branch
# ============================================================

class HyGAMChannelAttention(layers.Layer):
    """
    Hyperbolic Geometry Attention Mixer.

    This is the HySAR-compatible version of your HyGAM code.

    Input:
        x_i: [B, H, W, C]

    Output:
        A_NE_i: [B, 1, 1, C]

    This version uses:
        GAP + GMP + GSP + GMN descriptors
        low-rank projection
        Lorentz branch
        Poincare branch
        parallel geometry fusion
    """
    def __init__(
        self,
        units=None,
        learnable_curvature=True,
        reduction=8,
        min_rank=4,
        projection_activation=None,
        normalize_descriptor=True,
        normalize_geometry=True,
        use_scale=True,
        eps=1e-6,
        **kwargs
    ):
        super(HyGAMChannelAttention, self).__init__(**kwargs)

        self.units = None if units is None else int(units)
        self.learnable_curvature = bool(learnable_curvature)
        self.reduction = int(reduction)
        self.min_rank = int(min_rank)
        self.projection_activation = projection_activation
        self.normalize_descriptor = bool(normalize_descriptor)
        self.normalize_geometry = bool(normalize_geometry)
        self.use_scale = bool(use_scale)
        self.eps = float(eps)

        self.channels = None
        self.descriptor_dim = None

        self.descriptor_projection = None
        self.to_channels = None

        self.curv_log = None
        self.beta = None

        self.g_lorentz = None
        self.g_poincare = None
        self.logit_scale = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        self.descriptor_dim = self.units if self.units is not None else self.channels

        descriptor_rank = _rank_for(
            in_dim=self.channels,
            out_dim=self.descriptor_dim,
            reduction=self.reduction,
            min_rank=self.min_rank
        )

        self.descriptor_projection = LowRankDense(
            units=self.descriptor_dim,
            rank=descriptor_rank,
            activation=self.projection_activation,
            use_bias=False,
            name="hygam_descriptor_low_rank_projection"
        )

        if self.descriptor_dim != self.channels:
            back_rank = _rank_for(
                in_dim=self.descriptor_dim,
                out_dim=self.channels,
                reduction=self.reduction,
                min_rank=self.min_rank
            )

            self.to_channels = LowRankDense(
                units=self.channels,
                rank=back_rank,
                activation=None,
                use_bias=False,
                name="hygam_to_input_channels"
            )

        self.curv_log = self.add_weight(
            name="hygam_curv_log",
            shape=(),
            initializer=tf.keras.initializers.Zeros(),
            trainable=self.learnable_curvature
        )

        self.beta = self.add_weight(
            name="hygam_lorentz_beta",
            shape=(self.descriptor_dim,),
            initializer="ones",
            trainable=True
        )

        self.g_lorentz = self.add_weight(
            name="hygam_g_lorentz",
            shape=(),
            initializer="ones",
            trainable=True
        )

        self.g_poincare = self.add_weight(
            name="hygam_g_poincare",
            shape=(),
            initializer="ones",
            trainable=True
        )

        if self.use_scale:
            self.logit_scale = self.add_weight(
                name="hygam_logit_scale",
                shape=(1, self.channels),
                initializer="ones",
                trainable=True
            )

        super(HyGAMChannelAttention, self).build(input_shape)

    def call(self, inputs, training=None):
        # Diverse global descriptor statistics
        gap = tf.reduce_mean(inputs, axis=[1, 2])
        gmp = tf.reduce_max(inputs, axis=[1, 2])
        gsp = tf.reduce_sum(inputs, axis=[1, 2])
        gmn = tf.reduce_min(inputs, axis=[1, 2])

        descriptor = gap + gmp + gsp + gmn

        # Low-rank projection: lambda_i^j
        lam = self.descriptor_projection(descriptor)

        if self.normalize_descriptor:
            lam = _znorm_vector(lam, eps=self.eps)

        # Positive learnable curvature:
        # c = softplus(k) + eps
        c = tf.nn.softplus(self.curv_log) + 1e-3
        sqrt_c = tf.sqrt(c)

        # -----------------------------
        # Lorentz branch / LoMAR
        # -----------------------------
        norm_sq = tf.reduce_sum(tf.square(lam), axis=-1, keepdims=True)

        # Paper-style Lorentz time coordinate:
        # t(lambda) = sqrt(1 + c ||lambda||^2) / sqrt(c)
        t = tf.sqrt(1.0 + c * norm_sq) / (sqrt_c + self.eps)

        beta = tf.reshape(self.beta, [1, self.descriptor_dim])

        # Lorentz channel weights:
        # psi_L = -t^2 + beta * lambda
        lorentz = -tf.square(t) + beta * lam

        # -----------------------------
        # Poincare branch / PARL
        # -----------------------------
        norm = tf.norm(lam, axis=-1, keepdims=True) + self.eps

        # PoEM_c(lambda)
        poincare_embedding = (
            tf.tanh(sqrt_c * norm) * lam / ((sqrt_c * norm) + self.eps)
        )

        # psi_P = PoEM_c(lambda) * lambda
        poincare = poincare_embedding * lam

        # Stabilize geometry logits before fusion
        if self.normalize_geometry:
            lorentz = _znorm_vector(lorentz, eps=self.eps)
            poincare = _znorm_vector(poincare, eps=self.eps)

        logits = self.g_lorentz * lorentz + self.g_poincare * poincare

        if self.to_channels is not None:
            logits = self.to_channels(logits)

        if self.logit_scale is not None:
            logits = logits * self.logit_scale

        attention = tf.nn.sigmoid(logits)
        attention = tf.reshape(
            attention,
            [tf.shape(inputs)[0], 1, 1, self.channels]
        )

        return attention

    def get_config(self):
        config = super(HyGAMChannelAttention, self).get_config()
        config.update({
            "units": self.units,
            "learnable_curvature": self.learnable_curvature,
            "reduction": self.reduction,
            "min_rank": self.min_rank,
            "projection_activation": self.projection_activation,
            "normalize_descriptor": self.normalize_descriptor,
            "normalize_geometry": self.normalize_geometry,
            "use_scale": self.use_scale,
            "eps": self.eps,
        })
        return config


class HyperbolicDualGeometry(layers.Layer):
    """
    Backward-compatible HyGAM wrapper.

    Your pasted HyGAM returned one shared vector h.
    For HySAR, we need modality-specific non-Euclidean maps:

        A_NE_1 = HyGAM(x1)
        A_NE_2 = HyGAM(x2)

    Output:
        A_NE_1, A_NE_2
    """
    def __init__(
        self,
        units=128,
        learnable_curvature=True,
        reduction=8,
        use_scale=True,
        **kwargs
    ):
        super(HyperbolicDualGeometry, self).__init__(**kwargs)

        self.units = int(units)
        self.learnable_curvature = bool(learnable_curvature)
        self.reduction = int(reduction)
        self.use_scale = bool(use_scale)

        self.hygam1 = HyGAMChannelAttention(
            units=self.units,
            learnable_curvature=self.learnable_curvature,
            reduction=self.reduction,
            use_scale=self.use_scale,
            name="hygam_modality_1"
        )

        self.hygam2 = HyGAMChannelAttention(
            units=self.units,
            learnable_curvature=self.learnable_curvature,
            reduction=self.reduction,
            use_scale=self.use_scale,
            name="hygam_modality_2"
        )

    def call(self, inputs, training=None):
        x1, x2 = inputs
        a_ne_1 = self.hygam1(x1, training=training)
        a_ne_2 = self.hygam2(x2, training=training)
        return a_ne_1, a_ne_2

    def get_config(self):
        config = super(HyperbolicDualGeometry, self).get_config()
        config.update({
            "units": self.units,
            "learnable_curvature": self.learnable_curvature,
            "reduction": self.reduction,
            "use_scale": self.use_scale,
        })
        return config


# ============================================================
# Multi-Cross Fusion Module
# ============================================================

class HySARMultiCrossFusion2D(layers.Layer):
    """
    HySAR Multi-Cross Fusion for two modalities.

    Inputs:
        x1      : [B, H, W, C1]
        x2      : [B, H, W, C2]
        A_E_1   : [B, 1, 1, C1]  from MILA(x1)
        A_E_2   : [B, 1, 1, C2]  from MILA(x2)
        A_NE_1  : [B, 1, 1, C1]  from HyGAM(x1)
        A_NE_2  : [B, 1, 1, C2]  from HyGAM(x2)

    Cross fusion:
        For modality 1:
            own non-Euclidean cue: A_NE_1
            cross Euclidean cue : A_E_2 projected to C1

        For modality 2:
            own non-Euclidean cue: A_NE_2
            cross Euclidean cue : A_E_1 projected to C2

    mcf_mode:
        "path_softmax":
            practical two-path gate over {own HyGAM, cross MILA}

        "equation":
            stricter Eq. 18-style channel softmax after summing paths
    """
    def __init__(
        self,
        reduction=8,
        min_rank=4,
        mcf_mode="path_softmax",
        return_attention=True,
        **kwargs
    ):
        super(HySARMultiCrossFusion2D, self).__init__(**kwargs)

        if mcf_mode not in {"path_softmax", "equation"}:
            raise ValueError("mcf_mode must be either 'path_softmax' or 'equation'.")

        self.reduction = int(reduction)
        self.min_rank = int(min_rank)
        self.mcf_mode = mcf_mode
        self.return_attention = bool(return_attention)

        self.channels1 = None
        self.channels2 = None

        self.project_e2_to_1 = None
        self.project_e1_to_2 = None

        self.alpha_ne1 = None
        self.alpha_e2_to_1 = None
        self.alpha_out1 = None

        self.alpha_ne2 = None
        self.alpha_e1_to_2 = None
        self.alpha_out2 = None

    def build(self, input_shapes):
        x1_shape, x2_shape, ae1_shape, ae2_shape, ane1_shape, ane2_shape = input_shapes

        self.channels1 = int(x1_shape[-1])
        self.channels2 = int(x2_shape[-1])

        if self.channels2 != self.channels1:
            rank_21 = _rank_for(
                in_dim=self.channels2,
                out_dim=self.channels1,
                reduction=self.reduction,
                min_rank=self.min_rank
            )

            self.project_e2_to_1 = LowRankDense(
                units=self.channels1,
                rank=rank_21,
                use_bias=False,
                name="project_mila_modality_2_to_modality_1"
            )

        if self.channels1 != self.channels2:
            rank_12 = _rank_for(
                in_dim=self.channels1,
                out_dim=self.channels2,
                reduction=self.reduction,
                min_rank=self.min_rank
            )

            self.project_e1_to_2 = LowRankDense(
                units=self.channels2,
                rank=rank_12,
                use_bias=False,
                name="project_mila_modality_1_to_modality_2"
            )

        self.alpha_ne1 = self.add_weight(
            name="alpha_ne_modality_1",
            shape=(1, 1, 1, self.channels1),
            initializer="ones",
            trainable=True
        )

        self.alpha_e2_to_1 = self.add_weight(
            name="alpha_cross_euclidean_2_to_1",
            shape=(1, 1, 1, self.channels1),
            initializer="ones",
            trainable=True
        )

        self.alpha_out1 = self.add_weight(
            name="alpha_output_modality_1",
            shape=(1, 1, 1, self.channels1),
            initializer="ones",
            trainable=True
        )

        self.alpha_ne2 = self.add_weight(
            name="alpha_ne_modality_2",
            shape=(1, 1, 1, self.channels2),
            initializer="ones",
            trainable=True
        )

        self.alpha_e1_to_2 = self.add_weight(
            name="alpha_cross_euclidean_1_to_2",
            shape=(1, 1, 1, self.channels2),
            initializer="ones",
            trainable=True
        )

        self.alpha_out2 = self.add_weight(
            name="alpha_output_modality_2",
            shape=(1, 1, 1, self.channels2),
            initializer="ones",
            trainable=True
        )

        super(HySARMultiCrossFusion2D, self).build(input_shapes)

    def _project_attention(self, attention, projection, target_channels):
        if projection is None:
            return attention

        batch = tf.shape(attention)[0]
        v = tf.reshape(attention, [batch, -1])
        v = projection(v)
        return tf.reshape(v, [batch, 1, 1, target_channels])

    def _fuse_two_paths(self, own_ne_term, cross_e_term):
        if self.mcf_mode == "equation":
            z = own_ne_term + cross_e_term
            z = tf.nn.softmax(z, axis=-1)
            return tf.nn.sigmoid(z)

        # Practical HySAR two-path MCF:
        # softmax over {own HyGAM, cross MILA} for each channel.
        paths = tf.stack([own_ne_term, cross_e_term], axis=-2)
        gates = tf.nn.softmax(paths, axis=-2)
        fused = tf.reduce_sum(gates * paths, axis=-2)
        return tf.nn.sigmoid(fused)

    def call(self, inputs, training=None):
        x1, x2, a_e_1, a_e_2, a_ne_1, a_ne_2 = inputs

        # Cross-project Euclidean MILA maps when channel sizes differ.
        a_e_2_to_1 = self._project_attention(
            a_e_2,
            self.project_e2_to_1,
            self.channels1
        )

        a_e_1_to_2 = self._project_attention(
            a_e_1,
            self.project_e1_to_2,
            self.channels2
        )

        # Modality 1 hybrid attention:
        # own HyGAM(x1) + cross MILA(x2)
        term1_ne = self.alpha_ne1 * a_ne_1
        term1_cross_e = self.alpha_e2_to_1 * a_e_2_to_1
        a_h_1 = self._fuse_two_paths(term1_ne, term1_cross_e)

        # Modality 2 hybrid attention:
        # own HyGAM(x2) + cross MILA(x1)
        term2_ne = self.alpha_ne2 * a_ne_2
        term2_cross_e = self.alpha_e1_to_2 * a_e_1_to_2
        a_h_2 = self._fuse_two_paths(term2_ne, term2_cross_e)

        x_s_1 = x1 * a_h_1 * self.alpha_out1
        x_s_2 = x2 * a_h_2 * self.alpha_out2

        if self.return_attention:
            return x_s_1, x_s_2, a_h_1, a_h_2

        return x_s_1, x_s_2

    def get_config(self):
        config = super(HySARMultiCrossFusion2D, self).get_config()
        config.update({
            "reduction": self.reduction,
            "min_rank": self.min_rank,
            "mcf_mode": self.mcf_mode,
            "return_attention": self.return_attention,
        })
        return config


# ============================================================
# Complete HySAR block
# ============================================================

class HySARAttentionLayer(layers.Layer):
    """
    Complete HySAR shared representation refiner.

    Input:
        [input1, input2]

    Output:
        shared_feature1, shared_feature2

    Internally:
        A_E_1  = MILA(input1)
        A_E_2  = MILA(input2)

        A_NE_1 = HyGAM(input1)
        A_NE_2 = HyGAM(input2)

        output1, output2 = MCF(input1, input2, A_E_1, A_E_2, A_NE_1, A_NE_2)

    Notes:
        - units controls HyGAM descriptor width.
        - MILA remains channel-adaptive to the input feature channels.
        - stages=2 matches the paper's MILA implementation.
    """
    def __init__(
        self,
        units=128,
        stages=2,
        reduction=8,
        groups=8,
        use_scale=True,
        learnable_curvature=True,
        kernels=(1, 3, 5),
        dilation_rates=(1, 2, 3),
        ghost_ratio=2,
        ghost_kernel_size=3,
        mila_activation="relu",
        mcf_mode="path_softmax",
        return_attention=False,
        **kwargs
    ):
        super(HySARAttentionLayer, self).__init__(**kwargs)

        self.units = int(units)
        self.stages = int(stages)
        self.reduction = int(reduction)
        self.groups = int(groups)
        self.use_scale = bool(use_scale)
        self.learnable_curvature = bool(learnable_curvature)
        self.kernels = tuple(kernels)
        self.dilation_rates = tuple(dilation_rates)
        self.ghost_ratio = int(ghost_ratio)
        self.ghost_kernel_size = int(ghost_kernel_size)
        self.mila_activation = mila_activation
        self.mcf_mode = mcf_mode
        self.return_attention = bool(return_attention)

        # Euclidean branch: MILA
        self.mila1 = MILAChannelAttention(
            stages=self.stages,
            reduction=self.reduction,
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=self.dilation_rates,
            ghost_ratio=self.ghost_ratio,
            ghost_kernel_size=self.ghost_kernel_size,
            feature_activation=self.mila_activation,
            projection_activation=self.mila_activation,
            use_scale=self.use_scale,
            name="mila_modality_1"
        )

        self.mila2 = MILAChannelAttention(
            stages=self.stages,
            reduction=self.reduction,
            kernels=self.kernels,
            groups=self.groups,
            dilation_rates=self.dilation_rates,
            ghost_ratio=self.ghost_ratio,
            ghost_kernel_size=self.ghost_kernel_size,
            feature_activation=self.mila_activation,
            projection_activation=self.mila_activation,
            use_scale=self.use_scale,
            name="mila_modality_2"
        )

        # Non-Euclidean branch: HyGAM
        self.hygam1 = HyGAMChannelAttention(
            units=self.units,
            learnable_curvature=self.learnable_curvature,
            reduction=self.reduction,
            use_scale=self.use_scale,
            name="hygam_modality_1"
        )

        self.hygam2 = HyGAMChannelAttention(
            units=self.units,
            learnable_curvature=self.learnable_curvature,
            reduction=self.reduction,
            use_scale=self.use_scale,
            name="hygam_modality_2"
        )

        # Multi-cross fusion
        self.mcf = HySARMultiCrossFusion2D(
            reduction=self.reduction,
            mcf_mode=self.mcf_mode,
            return_attention=True,
            name="hysar_multi_cross_fusion"
        )

    def call(self, inputs, training=None):
        input1, input2 = inputs

        # MILA Euclidean attention maps
        a_e_1 = self.mila1(input1, training=training)
        a_e_2 = self.mila2(input2, training=training)

        # HyGAM non-Euclidean attention maps
        a_ne_1 = self.hygam1(input1, training=training)
        a_ne_2 = self.hygam2(input2, training=training)

        # MCF hybrid-space shared features
        x_s_1, x_s_2, a_h_1, a_h_2 = self.mcf(
            [input1, input2, a_e_1, a_e_2, a_ne_1, a_ne_2],
            training=training
        )

        if self.return_attention:
            return x_s_1, x_s_2, a_e_1, a_e_2, a_ne_1, a_ne_2, a_h_1, a_h_2

        return x_s_1, x_s_2

    def get_config(self):
        config = super(HySARAttentionLayer, self).get_config()
        config.update({
            "units": self.units,
            "stages": self.stages,
            "reduction": self.reduction,
            "groups": self.groups,
            "use_scale": self.use_scale,
            "learnable_curvature": self.learnable_curvature,
            "kernels": self.kernels,
            "dilation_rates": self.dilation_rates,
            "ghost_ratio": self.ghost_ratio,
            "ghost_kernel_size": self.ghost_kernel_size,
            "mila_activation": self.mila_activation,
            "mcf_mode": self.mcf_mode,
            "return_attention": self.return_attention,
        })
        return config


# ============================================================
# Backward-compatible replacement for your old DeeperAttentionLayer1
# ============================================================

class DeeperAttentionLayer1(HySARAttentionLayer):
    """
    Drop-in HySAR replacement for your previous EMCAM wrapper.

    Same usage:
        attention_feature1, attention_feature2 = DeeperAttentionLayer1(...)([input1, input2])
    """
    def __init__(
        self,
        units=128,
        use_scale=True,
        stages=2,
        reduction=8,
        groups=8,
        **kwargs
    ):
        super(DeeperAttentionLayer1, self).__init__(
            units=units,
            stages=stages,
            reduction=reduction,
            groups=groups,
            use_scale=use_scale,
            **kwargs
        )

In [ ]:
## MSTL
def residual_GLC_branch1(inputs1, inputs2):
    
    x1 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs1)
    #x1 = DeeperAttentionLayer1(units=64, use_scale=True)(x1) ## MFA ####
    #x1 = AttentionBlock(64)(x1)
    x1 = BatchNormalization()(x1)
    x1 = tf.keras.layers.Activation('relu')(x1)
    x1 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x1)
    
    x2 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs2)
    #x2 = DeeperAttentionLayer1(units=64, use_scale=True)(x2) ## MFA ####
    #x2 = AttentionBlock(64)(x2)
    x2 = BatchNormalization()(x2)
    x2 = tf.keras.layers.Activation('relu')(x2)
    x2 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x2)
    
    ## ERLA
    x1 = RGSA(x1, filters=64)
    #
    x2 = RGSA(x2, filters=64)
    
    
    
    
    x1 = RGSA(x1, filters=64)
    
    x2 = RGSA(x2, filters=64)
    
    #x1, x2 = DeeperAttentionLayer1(units=64, use_scale=True)([x1, x2])

    '''mila_layer = DeeperAttentionLayer1(units=64, stages=2, reduction=4, groups=8, use_scale=True)
    x1, x2 = mila_layer([x1, x2])
    '''

    hysar_layer = HySARAttentionLayer(
    units=64,
    stages=2,
    reduction=4,
    groups=8,
    use_scale=True)
    x1, x2 = hysar_layer([x1, x2])
    
    
    ## ERLA
    x1 = RGSA(x1, filters=128, strides=(2, 2), use_projection=True)
    
    x2 = RGSA(x2, filters=128, strides=(2, 2), use_projection=True)
    
    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    
    x1 = RGSA(x1, filters=128)
    
    x2 = RGSA(x2, filters=128)

    
    #x1, x2 = DeeperAttentionLayer1(units=128, use_scale=True)([x1, x2])

    """mila_layer = DeeperAttentionLayer1(units=128, stages=2, reduction=4, groups=8, use_scale=True)
    x1, x2 = mila_layer([x1, x2])
    """

    hysar_layer = HySARAttentionLayer(
    units=128,
    stages=2,
    reduction=4,
    groups=8,
    use_scale=True)
    x1, x2 = hysar_layer([x1, x2])
    
    
    ## ERLA
    x1 = RGSA(x1, filters=256, strides=(2, 2), use_projection=True)
    
    
    x2 = RGSA(x2, filters=256, strides=(2, 2), use_projection=True)
    

    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    
    x1 = RGSA(x1, filters=256)
    
    x2 = RGSA(x2, filters=256)
    
    #x1, x2 = DeeperAttentionLayer1(units=256, use_scale=True)([x1, x2])

    '''mila_layer = DeeperAttentionLayer1(units=256, stages=2, reduction=8, groups=16, use_scale=True)
    x1, x2 = mila_layer([x1, x2])
    '''

    hysar_layer = HySARAttentionLayer(
    units=256,
    stages=2,
    reduction=8,
    groups=16,
    use_scale=True)
    x1, x2 = hysar_layer([x1, x2])
    
    
    ## ERLA
    x1 = RGSA(x1, filters=512, strides=(2, 2), use_projection=True)
    
    x2 = RGSA(x2, filters=512, strides=(2, 2), use_projection=True)

    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    ## ERLA
    x1 = RGSA(x1, filters=512)
    x2 = RGSA(x2, filters=512)

    #x1, x2 = DeeperAttentionLayer1(units=512, use_scale=True)([x1, x2])

    '''mila_layer = DeeperAttentionLayer1(units=512, stages=2, reduction=8, groups=16, use_scale=True)
    x1, x2 = mila_layer([x1, x2])'''

    hysar_layer = HySARAttentionLayer(
    units=512,
    stages=2,
    reduction=8,
    groups=16,
    use_scale=True)
    x1, x2 = hysar_layer([x1, x2])

    
    
    return x1, x2

In [ ]:
#def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
input_shape=(128, 128, 3)
inputs1 = Input(shape=input_shape)
inputs2 = Input(shape=input_shape)

import tensorflow.keras.layers as L

#input_data = Input(shape=input_shape, name='input_data')
# Initial convolutional layer

x1, x2 = residual_GLC_branch1(inputs1, inputs2)
#print('x:',x.shape)


con = tf.keras.layers.Concatenate(axis=-1)([x1, x2])

#con = tf.keras.layers.Dropout(0.25)(con) 

x = GlobalAveragePooling2D()(con)
#print('GlobalAveragePooling2D x:',x.shape)

## TMTL
outputs1 = Dense(5, activation='softmax')(x)
outputs2 = Dense(7, activation='softmax')(x)

# Create the model
model = Model([inputs1, inputs2], [outputs1, outputs2])
#return model
print(model.summary())

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Add, Concatenate

def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(
        [tf.TensorSpec([1, *inputs.shape[1:]]) for inputs in model.inputs]
    )
    
    # Calculate FLOPs using TensorFlow's built-in profiler
    graph = concrete_func.graph
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    
    flops = tf.compat.v1.profiler.profile(
        graph,
        run_meta=run_meta,
        cmd='op',
        options=opts
    )
    
    return flops.total_float_ops

# Load your trained model or build it using the provided code
# model = ... (your model loading/building code here)

# Calculate total FLOPs
total_flops = get_flops(model)
gflops = total_flops / 1e9

print(f"Total FLOPs: {total_flops:,}")
print(f"GFLOPs: {gflops:.2f}")

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay

initial_gamma = 0.5
learning_rate = 1e-2
optimizer = Adam(learning_rate=0.0001)
#opt = Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.9, epsilon=None, amsgrad=False)
# Compile the model with the custom optimizer
model.compile(optimizer=optimizer,
              loss=['categorical_crossentropy', 'categorical_crossentropy'],
              loss_weights=[initial_gamma, (1 -  initial_gamma)],
              metrics=['accuracy', 'accuracy'])


from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best1_model_cer_skin_lung.keras'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           #frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                            mode='min',
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=80, verbose=1)
    return es_callback



from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1,
                              patience=70, verbose = 1, min_lr=0.00001)

checkpoint_callback = checkpoint_callback()

early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping, reduce_lr]

In [ ]:
# Fit the model with callbacks
history = model.fit([X_train_s, X_train_h], [y_train_s, y_train_h],
                    epochs=200,
                    validation_data=([X_val_s, X_val_h1], [y_val_s, y_val_h1]), verbose=1,
                    shuffle=True,
                    callbacks=callbacks) # UpdateGammaCallback

In [ ]:
model.evaluate([X_test_s, X_test_h1], [y_test_s, y_test_h1])